# CayleyPy AZ4 Trainer - Megaminx

Train **AZ v4**, a strong 6M-param value+policy model for the
[CayleyPy Megaminx competition](https://www.kaggle.com/competitions/cayley-py-megaminx) -
end-to-end, with every stage's hyperparameters exposed in one config cell.

On a 51-puzzle stratified sample (beam 65536, our production solver), the extracted AZ v4
value head solves **51/51 with mean path 87.5**, vs 89.4 for the previous best pure-V model
of the same size. It was the first 6M model in our pipeline to break the "89-moves cluster"
without inference-side tricks.

This notebook mirrors the structure of
[CayleyPy RW-ModelBaselines Megaminx AdamW](https://www.kaggle.com/code/ogurtsov/cayleypy-rw-modelbaselines-megaminx-adamw)
by @ogurtsov (config-first, modular, ends with beam search + submission), but replaces the
single random-walk regression stage with the full 5-stage AZ v4 pipeline. The **model pools
of both notebooks are merged**: `CFG['model']['model_type']` switches between our ResMLP
(the AZ v4 architecture) and the community `PilgrimAttnRes` family - and any registered
model can be trained through any subset of the stages. Model training is self-contained
PyTorch (all code inlined below); beam search / solving uses the
[cayleypy](https://github.com/cayleypy/cayleypy) library, so results are directly comparable
with the other community baselines.

## The 5-stage pipeline

```
stage 1 "pretrain"    random-walk MSE distance regression        4000 ep   (from scratch)
stage 2 "curriculum"  mixed-k curriculum + EMA + val early-stop  ~6k ep    (warm from 1)
stage 3 "bellman"     Bellman bootstrap + frontier + BFS-d6      500 ep    (warm from 2)
stage 4 "bellman_dd"  + exact anchors at solved / depth-1        50 ep     (warm from 3)
stage 5 "az"          dual-head: policy CE + Bellman value       ~25 ep    (warm from 4)
                      -> export value head = the beam-search model
```

Why so many stages? Each fixes a failure mode of the previous one:

- **pretrain** learns "random-walk depth", an upper bound on true distance.
- **curriculum** improves mid-depth calibration by mixing walk lengths (and EMA smooths it).
- **bellman** replaces the walk-depth label with the self-consistent target
  `1 + min_a V(child)` (DeepCubeA-style), plus two data mixins: 25% beam-frontier states
  (the distribution beam search actually visits) and 10% exact-distance states (all 19.3M
  states within 6 moves of solved).
- **bellman_dd** anchors `V(solved)=0` and `V(depth-1)=1` exactly in every batch (the
  bootstrap alone leaves V(solved) near 2).
- **az** adds a policy head (cross-entropy on a strong community solutions set) while
  keeping the value recipe - the shared trunk learns from both signals, and the VALUE head
  comes out better than any pure-V model we trained. Counter-intuitively you must stop
  early (~epoch 24): past that the trunk memorizes the policy data and the value head
  degrades. Watch the printed top-1 accuracy - when it passes ~35-40% you are past the
  sweet spot.

**Default run: the FULL pipeline, from scratch.** The whole chain needs ~25-30h on a T4 -
several Kaggle sessions - so the notebook stops training gracefully at a wall budget
(default 8.5h), saves all checkpoints plus a `models/chain_state.json` progress marker,
and **the next run of the same notebook continues automatically from its own previous
output** (which is attached as an input). Just press "Save & Run All" again until the log
says `CHAIN COMPLETE`; the final session trains the az stage, exports the value head, and
runs the calibration + beam checks.

**Quick mode**: set `stages_to_run = ['az']` to skip stages 1-4 and warm-start from the
shipped stage-4 checkpoint instead - that reproduces AZ v4 in ~30 minutes on a T4.

Assets (checkpoints + datasets) are in
[megaminx-az4-training-assets](https://www.kaggle.com/datasets/artgor/megaminx-az4-training-assets).
Add it via "Add Input" if it is not already attached, together with the competition data.

# Params

Everything is driven by the single `CFG` dict in the next cell. Groups:

- **manage**: `stages_to_run` (any contiguous subset of
  `['pretrain','curriculum','bellman','bellman_dd','az']`; default = all five),
  `quick_test` (tiny smoke run), `experiment`, `seed`, `precision`, `compile`.

  **`experiment` is the chain identity.** Multi-session runs resume only a previous
  chain whose `(experiment, model, seed)` all match - to start a NEW training from
  scratch, just give it a new `experiment` label; the old chain is left alone and
  ignored. Ordinary stage hyperparameters (lr, epochs, mixin fractions, ...) are
  deliberately NOT part of the identity: change only a later stage and the chain
  correctly reuses your completed earlier stages. Convention: one experiment label =
  one recipe.
- **multi-session chaining**: `max_wall_hours` / `eval_reserve_minutes` / `auto_resume`.
  A Kaggle run killed by the session limit saves NO output, so training stops gracefully
  at the wall budget instead, writes every checkpoint plus `models/chain_state.json`, and
  the next run continues from its own previous output (the notebook's output is attached
  as one of its inputs). Progress carries across runs per stage AND per epoch within a
  stage; the chain state is keyed to (model_type, params, seed) - change any of those and
  it starts fresh.
- **model (the MODEL POOL)**: `CFG['model']['model_type']` selects the backbone, and each
  registered model keeps its own parameter block in the same dict. Out of the box:
  - `'ResMLPDistance'` - our architecture (embedding + LayerNorm ResMLP). The default
    params `(2048, 512) x 2 blocks, embed 16` = the 6.0M AZ v4 trunk.
  - `'PilgrimAttnRes'` - the community baseline family from @ogurtsov's modular notebook
    (one-hot + BatchNorm; `block_type='residual'` or `'attn_res'`).

  **Any model runs through ALL five stages**: stages 1-4 train it as a plain distance
  model, and stage 5 wraps the same backbone with policy + value heads. To add your own:
  define the class in the model-pool cell (needs `forward`, `features`, `feature_dim`),
  register it in `MODEL_REGISTRY`, and add its parameter block under `CFG['model']`.

  Two caveats: (1) the shipped warm-start checkpoints exist only for the default 6M
  ResMLP - any other model/shape must be trained from `'pretrain'` up (the notebook
  enforces this with a clear error); (2) in our experiments bigger ResMLP trunks
  REGRESSED under this exact recipe (a 20.5M model lost to the 6M one at every
  checkpoint), and the whole recipe was tuned on the ResMLP - treat other backbones as
  experiments and judge them with the canary + bench cells, not the training loss.
- **optimizer / loss factories** (same pattern as the community notebook): each stage has
  `'optimizer': {'name': ..., 'params': {...}}` resolving any `torch.optim` class (lr
  still comes from the stage's `'lr'` key), and the two regression stages
  (`pretrain`/`curriculum`) also take `'loss': {'name': ..., 'params': {...}}` - any
  `torch.nn` loss plus `PinballLoss` / `LogCoshLoss`. The Bellman/az losses are part of
  the algorithm and stay fixed.
- **per-stage sections** (`pretrain`, `curriculum`, `bellman`, `bellman_dd`, `az`): epochs,
  batch sizes, learning rates, mixin fractions, anchor counts, target-net refresh cadence.
  Defaults are the exact values that produced AZ v4. Each stage's `warmstart` is `'auto'`
  (previous stage's fresh output if you trained it this session, else the shipped
  checkpoint) or a path.
- **eval**: calibration canary + small cayleypy beam benchmark.
- **solve**: full solving block writing `submission.csv`. Off by default;
  `list_states_to_solve` is a short list by default - set it to `[]` to solve ALL 1001
  puzzles (many hours at useful beam widths).

Approximate wall-clock per stage (Kaggle T4, fp32; the az number is measured, the rest
are scaled from our local runs; local RTX 4090 with bf16+compile is ~4x faster):

| stage | epochs | est. T4 wall |
|---|---|---|
| pretrain | 4000 | ~6-8 h |
| curriculum | early-stops ~6k | ~7-10 h |
| bellman | 500 | ~8-12 h |
| bellman_dd | 50 | ~1-1.5 h |
| az (+ eval) | 30 | ~0.5 h (measured) |
| full chain | | ~25-30 h = 3-4 sessions via auto-resume |

Key facts baked into the defaults (learned the hard way):

- **Training loss is NOT the model-selection signal.** For the `az` stage the best value
  head is at ~epoch 24 even though both losses keep "improving" long after. Checkpoints are
  saved every `checkpoint_every` epochs; the canary + bench cells below are the judge.
- **`az.lr_t_max` stays 200 even though default `epochs=30`.** The original run used a
  200-epoch cosine schedule and early-stopped; shortening T_max would change the LR at
  epoch 24 and give a different model.
- **Precision `'auto'` = bf16 only on Ampere+ GPUs, else fp32.** P100 has no bf16; T4
  emulates it with a catastrophic slowdown (we measured ~40x on inference). fp32 is fine
  for a 6M model.
- **`rw_batch_size=8192 / policy_batch_size=1024` matter.** Running the az stage at half
  batch (the old script defaults) gives 4x the gradient steps per epoch and the policy
  memorizes before the value head is ready.

In [ ]:
### CONFIG - the whole notebook is driven by this dict ###
import json
import os

CFG = {
    # ---- manage ----
    # Any contiguous subset of ['pretrain','curriculum','bellman','bellman_dd','az'].
    # Default = the FULL from-scratch chain (~25-30h on a T4 - several sessions!).
    # Set ['az'] to just reproduce AZ v4 from the shipped stage-4 checkpoint (~30 min).
    'stages_to_run': ['pretrain', 'curriculum', 'bellman', 'bellman_dd', 'az'],
    'quick_test': False,      # True = tiny epochs/samples everywhere (smoke test)
    # Experiment label. Part of the chain identity: runs resume ONLY a chain whose
    # (experiment, model, seed) all match. Give every new training/recipe its own
    # label to start a fresh chain - no need to touch the seed or the inputs.
    'experiment': 'az4-baseline',
    'seed': 71,
    'precision': 'auto',      # 'auto' | 'off' (fp32) | 'bf16'
    'compile': 'auto',        # 'auto' (Ampere+ only) | True | False

    # Multi-session chaining: a Kaggle run that hits the session limit loses ALL its
    # output, so training stops GRACEFULLY at this wall budget (checkpoints +
    # models/chain_state.json land in the output), and the next run of this notebook
    # continues from its own previous output automatically (auto_resume; the notebook's
    # own output must be attached as an input - the published version already has it).
    # Just re-run the notebook until the chain reports all stages complete.
    'max_wall_hours': 8.5,        # keep safely under the Kaggle session limit
                                  # (normal value: 8.5 - keep safely under the session limit)
    'eval_reserve_minutes': 20,   # left aside for export + canary + bench + saving
    'auto_resume': True,

    # ---- MODEL POOL (the selected backbone is used by ALL stages) ----
    # 'model_type' picks the architecture; each registered model keeps its own parameter
    # block below. Add your own model in the model-pool cell and list its params here.
    'model': {
        'model_type': 'ResMLPDistance',
        'ResMLPDistance': {           # our architecture; these defaults = the AZ v4 6M trunk
            'hidden_dims': (2048, 512),   # see the bigger-trunk warning in the Params cell
            'num_res_blocks': 2,
            'embed_dim': 16,
        },
        'PilgrimAttnRes': {           # community baseline family (@ogurtsov's notebook)
            'hd1': 1024,
            'hd2': 256,
            'nrd': 4,                     # number of plain residual blocks
            'n_attn_blocks': 0,           # used when block_type='attn_res'
            'block_type': 'residual',     # 'residual' | 'attn_res'
            'dropout_rate': 0.0,
        },
    },

    # ---- stage 1: random-walk regression pretrain (from scratch) ----
    'pretrain': {
        'warmstart': None,            # from scratch
        'epochs': 4000,
        'samples_per_epoch': 1_000_000,
        'batch_size': 16384,
        'k_max': 80,                  # walk length; ~ puzzle diameter x 3
        'n_back': 1,                  # non-backtracking depth
        'lr': 2e-3,
        'optimizer': {'name': 'AdamW', 'params': {}},   # any torch.optim name; lr comes from 'lr'
        'loss': {'name': 'MSELoss', 'params': {}},      # torch.nn loss or PinballLoss/LogCoshLoss
        'checkpoint_every': 200,
        'resume': None,               # path to a checkpoint to continue from
    },

    # ---- stage 2: mixed-k curriculum + EMA + val early-stop ----
    'curriculum': {
        'warmstart': 'auto',
        'budget_epochs': 50_000,      # early stopping usually fires near ~6k
        'warmup_epochs': 5_000,
        'warmup_k_max': 35,
        'mix_k_list': (50, 70, 80, 100),
        'samples_per_epoch': 300_000,
        'batch_size': 8192,
        'lr': 5e-4,
        'n_back': 1,
        'optimizer': {'name': 'AdamW', 'params': {}},
        'loss': {'name': 'MSELoss', 'params': {}},      # training loss; val metric stays MSE
        'ema_tau': 1e-3,              # per-step EMA decay; best_ema.pt is the output
        'val_every': 100,
        'patience': 1000,             # early stop: no val improvement for this many epochs
        'val_k_max': 80,
        'val_n_walks': 2000,
        'checkpoint_every': 500,
        'resume': None,
    },

    # ---- stage 3: Bellman bootstrap + frontier + BFS-d6 mixins ----
    'bellman': {
        'warmstart': 'auto',
        'epochs': 500,
        'samples_per_epoch': 500_000,
        'batch_size': 8192,
        'k_max': 80,
        'n_back': 1,
        'lr': 5e-4,
        'optimizer': {'name': 'AdamW', 'params': {}},
        'target_update_every': 10,    # copy model -> frozen target net every N epochs
        'target_net_chunk': 4096,
        'clip_upper': True,           # clip target at walk depth (a provable upper bound)
        'clip_lower': True,           # clip target at 0
        'frontier_fraction': 0.25,    # beam-frontier states mixin (Bellman-bootstrap targets)
        'frontier_walk_depth_cap': 200.0,
        'bfs_d6_fraction': 0.10,      # exact-distance states (d<=6) mixin
        'n_anchor_v0': 0,             # stage 3 has NO anchors (stage 4 adds them)
        'n_anchor_d1': 0,
        'checkpoint_every': 25,
        'resume': None,
    },

    # ---- stage 4: + exact anchors (V(solved)=0, V(d=1)=1 every batch) ----
    'bellman_dd': {
        'warmstart': 'auto',
        'epochs': 50,
        'samples_per_epoch': 500_000,
        'batch_size': 8192,
        'k_max': 80,
        'n_back': 1,
        'lr': 5e-4,
        'optimizer': {'name': 'AdamW', 'params': {}},
        'target_update_every': 10,
        'target_net_chunk': 4096,
        'clip_upper': True,
        'clip_lower': True,
        'frontier_fraction': 0.25,
        'frontier_walk_depth_cap': 200.0,
        'bfs_d6_fraction': 0.10,
        'n_anchor_v0': 32,            # 32 copies of solved with target 0, every batch
        'n_anchor_d1': 4,             # 4 copies of each of the 24 depth-1 states, target 1
        'checkpoint_every': 10,
        'resume': None,
        # Note: the original stage-4 run also used a pattern-database lower-bound penalty
        # (lambda_pdb=5). Omitted here (PDB files are ~1.8GB and the effect is secondary);
        # the shipped stage-4 checkpoint IS the original lambda=5 artifact.
    },

    # ---- stage 5: dual-head AZ fine-tune (policy CE + Bellman value) ----
    'az': {
        'warmstart': 'auto',
        # Policy data: a solutions CSV (initial_state_id,path). 'auto' = the shipped
        # community-best submission_73731.csv. Point this at any stronger CSV to retrain
        # on better paths - more-consistent merged paths converge FASTER (stop earlier!).
        'policy_csv': 'auto',
        'policy_dataset': 'auto',     # 'auto' = prebuilt tensor for the default CSV, else built from policy_csv
        'epochs': 30,                 # original budget was 200; the good model is ~ep24
        'select_epoch': 24,           # checkpoint used for export/eval ('last' = final epoch)
        'lr_t_max': 200,              # cosine schedule length - keep 200 (see Params cell)
        'samples_per_epoch': 500_000,
        'rw_batch_size': 8192,        # value-batch size (random walks + anchors + BFS-d6)
        'policy_batch_size': 1024,    # policy CE batch per step
        'alpha': 1.0,                 # policy CE weight
        'beta': 1.0,                  # value MSE weight
        'k_max': 80,
        'n_back': 1,
        'lr': 5e-4,
        'optimizer': {'name': 'AdamW', 'params': {}},
        'anchor_v0': 32,
        'anchor_d1': 4,
        'bfs_d6_fraction': 0.10,
        'target_update_every': 10,
        'checkpoint_every': 5,        # ep 4,9,14,19,24,29 - explore stops near 24
        'resume': None,
    },

    # ---- eval ----
    'eval': {
        'canary': True,               # V(solved), V(d=1), exact-distance MAE, saturation probe
        'canary_bfs_samples': 200_000,
        'compare_reference': True,    # also canary the shipped m_az_v4_v_only for side-by-side
        'bench': True,                # small cayleypy beam benchmark on a few puzzles
        'bench_pids': (0, 100, 300),
        'bench_beam_width': 2 ** 12,
        'bench_max_steps': 200,
        'history_depth': 10,          # non-backtracking depth for cayleypy beam
    },

    # ---- solve + submission.csv (off by default) ----
    'solve': {
        'enabled': False,
        'checkpoint': 'auto',         # 'auto' = this run's exported value head, else shipped reference
        'list_states_to_solve': list(range(0, 20)),  # SHORT list by default; [] = ALL 1001 (hours!)
        'beam_width': 2 ** 14,
        'max_steps': 400,
    },
}

# Env-var overrides (used by the local smoke harness; harmless on Kaggle).
if os.environ.get('AZ4_QUICK_TEST') == '1':
    CFG['quick_test'] = True
if os.environ.get('AZ4_STAGES'):
    CFG['stages_to_run'] = [s for s in os.environ['AZ4_STAGES'].split(',') if s]
if os.environ.get('AZ4_EXPERIMENT'):
    CFG['experiment'] = os.environ['AZ4_EXPERIMENT']
if os.environ.get('AZ4_MODEL_TYPE'):
    CFG['model']['model_type'] = os.environ['AZ4_MODEL_TYPE']
if os.environ.get('AZ4_SOLVE') == '1':
    CFG['solve']['enabled'] = True

STAGE_ORDER = ['pretrain', 'curriculum', 'bellman', 'bellman_dd', 'az']
assert all(s in STAGE_ORDER for s in CFG['stages_to_run']), CFG['stages_to_run']

if CFG['quick_test']:
    print('QUICK TEST MODE: tiny epochs/samples everywhere - results are meaningless.')
    CFG['compile'] = False
    CFG['max_wall_hours'] = 99.0
    CFG['pretrain'].update(epochs=2, samples_per_epoch=20_000, batch_size=4096,
                           checkpoint_every=2)
    CFG['curriculum'].update(budget_epochs=3, warmup_epochs=1, patience=2, val_every=1,
                             samples_per_epoch=20_000, val_n_walks=50, checkpoint_every=2)
    CFG['bellman'].update(epochs=2, samples_per_epoch=20_000, batch_size=2048,
                          checkpoint_every=2)
    CFG['bellman_dd'].update(epochs=2, samples_per_epoch=20_000, batch_size=2048,
                             checkpoint_every=2)
    CFG['az'].update(epochs=2, select_epoch=1, samples_per_epoch=20_000,
                     rw_batch_size=2048, policy_batch_size=256, checkpoint_every=1)
    CFG['eval'].update(canary_bfs_samples=20_000, bench_pids=(0,),
                       bench_beam_width=2 ** 8, bench_max_steps=60)
    CFG['solve'].update(list_states_to_solve=[0], beam_width=2 ** 8, max_steps=60)

if os.environ.get('AZ4_MAX_WALL_HOURS'):        # local smoke-test hook
    CFG['max_wall_hours'] = float(os.environ['AZ4_MAX_WALL_HOURS'])

with open('cfg.json', 'w', encoding='utf-8') as f:
    json.dump(CFG, f, indent=2, default=list)
print('stages_to_run:', CFG['stages_to_run'])

In [ ]:
# cayleypy is used ONLY for the beam-search benchmark / solve cells at the end
# (training below is self-contained PyTorch). Same pinned commit as the other
# community baseline notebooks.
import importlib.util

if importlib.util.find_spec('cayleypy') is None:
    get_ipython().system('pip install git+https://github.com/cayleypy/cayleypy.git@e1518b6 --no-deps')
else:
    print('cayleypy already installed')

In [ ]:
### Environment: device, precision, compile ###
import contextlib
import time

import numpy as np
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', DEVICE)
if DEVICE == 'cuda':
    print(' ', torch.cuda.get_device_name(0))
    _CAP = torch.cuda.get_device_capability()
    # Recent Kaggle images ship torch builds WITHOUT Pascal (sm_60 / P100) kernels -
    # every CUDA op then fails with "no kernel image is available". Probe once and
    # fail with a clear message instead of crashing mid-training.
    try:
        (torch.ones(2, device='cuda') + 1).sum().item()
    except Exception as e:
        raise RuntimeError(
            f'This torch build cannot run on {torch.cuda.get_device_name(0)} '
            f'(capability {_CAP}). On Kaggle: Settings -> Accelerator -> GPU T4 x2, '
            f'then re-run. Original error: {e}') from None
else:
    _CAP = (0, 0)


def _resolve_precision() -> str:
    p = CFG['precision']
    if p != 'auto':
        return p
    # bf16 only on Ampere+ (P100 has no bf16; T4 emulates it ~40x slower).
    if DEVICE == 'cuda' and _CAP[0] >= 8 and torch.cuda.is_bf16_supported():
        return 'bf16'
    return 'off'


PRECISION = _resolve_precision()
COMPILE = CFG['compile'] if CFG['compile'] != 'auto' else (DEVICE == 'cuda' and _CAP[0] >= 8)
# Fused AdamW: safe on Volta+; skip on Pascal (P100).
FUSED_OK = DEVICE == 'cuda' and _CAP[0] >= 7
print(f'precision: {PRECISION} | compile: {COMPILE} | fused optimizer: {FUSED_OK}')


def make_autocast():
    if PRECISION == 'bf16' and DEVICE == 'cuda':
        return torch.amp.autocast('cuda', dtype=torch.bfloat16)
    return contextlib.nullcontext()


def make_optimizer(params, stage_cfg):
    """Name-based optimizer factory (same pattern as the community baselines notebook).

    The learning rate always comes from the stage's 'lr' key; extra constructor kwargs
    from stage_cfg['optimizer']['params'] (e.g. {'weight_decay': 0.01, 'betas': ...}).
    """
    opt_cfg = stage_cfg.get('optimizer') or {'name': 'AdamW', 'params': {}}
    name = opt_cfg.get('name') or 'AdamW'
    cls = getattr(torch.optim, name, None)
    if cls is None:
        raise ValueError(f'unknown optimizer {name!r} (expected a torch.optim name)')
    kw = dict(opt_cfg.get('params') or {})
    kw.setdefault('weight_decay', 0.0)
    if name in ('AdamW', 'Adam') and FUSED_OK:
        kw.setdefault('fused', True)
    return cls(params, lr=stage_cfg['lr'], **kw)


def maybe_compile(model):
    if COMPILE:
        return torch.compile(model, dynamic=False)
    return model


torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

# Wall-clock budget: training stops gracefully at TRAIN_DEADLINE so the run COMPLETES
# (a Kaggle run killed by the session limit saves no output at all).
NOTEBOOK_T0 = time.time()
TRAIN_DEADLINE = (NOTEBOOK_T0 + CFG['max_wall_hours'] * 3600
                  - CFG['eval_reserve_minutes'] * 60)


def training_time_up():
    return time.time() >= TRAIN_DEADLINE


print(f"wall budget: {CFG['max_wall_hours']}h "
      f"(training stops {CFG['eval_reserve_minutes']}min early for eval + saving)")

In [ ]:
### Inputs: competition data + assets dataset + working dirs ###
from pathlib import Path

ON_KAGGLE = Path('/kaggle/input').exists()

if ON_KAGGLE:
    _comp_candidates = [
        Path('/kaggle/input/competitions/cayley-py-megaminx'),
        Path('/kaggle/input/cayley-py-megaminx'),
    ]
    COMP = next((p for p in _comp_candidates if (p / 'puzzle_info.json').exists()), None)
    assert COMP is not None, (
        'Competition data not found. Add Input -> Competitions -> cayley-py-megaminx')
    _asset_candidates = [
        Path('/kaggle/input/megaminx-az4-training-assets'),
        Path('/kaggle/input/datasets/artgor/megaminx-az4-training-assets'),
        Path('/kaggle/input/datasets/megaminx-az4-training-assets'),
    ]
    ASSETS = next((p for p in _asset_candidates if p.exists()), None)
    if ASSETS is None:
        print('contents of /kaggle/input:',
              sorted(str(p) for p in Path('/kaggle/input').glob('**/*'))[:40])
        raise AssertionError(
            'Assets dataset not mounted. Add Input -> Datasets -> '
            'artgor/megaminx-az4-training-assets (and re-run).')
    WORK = Path('/kaggle/working')
else:
    # Local run: point these env vars at directories with the same files.
    COMP = Path(os.environ.get('AZ4_COMP_DIR', './comp'))
    ASSETS = Path(os.environ.get('AZ4_ASSETS_DIR', './assets'))
    WORK = Path(os.environ.get('AZ4_WORK_DIR', './az4_work'))
    WORK.mkdir(parents=True, exist_ok=True)

PUZZLE_JSON = COMP / 'puzzle_info.json'
TEST_CSV = COMP / 'test.csv'
SAMPLE_SUB = COMP / 'sample_submission.csv'
MODELS_DIR = WORK / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Shipped warm-start checkpoints: stage name -> the file that stage warm-starts FROM
# (i.e. the previous stage's output, from our original AZ v4 run).
STAGE_INPUT_ASSET = {
    'curriculum': 'm07_big_k80_epoch3999.pt',
    'bellman': 'm_curr_v0_best_ema.pt',
    'bellman_dd': 'm_curr_v3_epoch0499.pt',
    'az': 'm_dd_v0_epoch0049.pt',
}
REFERENCE_V_ONLY = ASSETS / 'm_az_v4_v_only.pt'       # our original exported value head
BFS_D6_PATH = ASSETS / 'bfs_d6_train.pt'              # 19.35M exact-distance states, ~2.3GB
FRONTIER_PATH = ASSETS / 'frontier_states.pt'         # 300K beam-frontier states
DEFAULT_POLICY_CSV = ASSETS / 'submission_73731.csv'  # community-best public solutions
PREBUILT_POLICY_DATASET = ASSETS / 'az_dataset_73731.pt'

for p in (PUZZLE_JSON, TEST_CSV, SAMPLE_SUB):
    assert p.exists(), f'missing input file: {p}'

# Output of a PREVIOUS run of this notebook (for auto-continuation of multi-session
# chains). Found via its models/chain_state.json marker.
PREV_OUTPUT = None
if ON_KAGGLE:
    _prev_candidates = [
        Path('/kaggle/input/notebooks/artgor/cayleypy-az4-trainer-megaminx'),
        Path('/kaggle/input/cayleypy-az4-trainer-megaminx'),
        Path('/kaggle/input/kernels/artgor/cayleypy-az4-trainer-megaminx'),
    ]
    PREV_OUTPUT = next((c for c in _prev_candidates
                        if (c / 'models' / 'chain_state.json').exists()), None)
    if PREV_OUTPUT is None:  # mount conventions move around - scan shallowly
        for _pat in ('*', '*/*', '*/*/*'):
            _hits = [d for d in Path('/kaggle/input').glob(_pat)
                     if d.is_dir() and (d / 'models' / 'chain_state.json').exists()]
            if _hits:
                PREV_OUTPUT = _hits[0]
                break
elif os.environ.get('AZ4_PREV_OUTPUT'):
    _c = Path(os.environ['AZ4_PREV_OUTPUT'])
    PREV_OUTPUT = _c if (_c / 'models' / 'chain_state.json').exists() else None

print('competition data:', COMP)
print('assets:', ASSETS, '| assets present:',
      all(p.exists() for p in (BFS_D6_PATH, FRONTIER_PATH, DEFAULT_POLICY_CSV)))
print('previous run output:', PREV_OUTPUT if PREV_OUTPUT else 'none found')

In [ ]:
### Library code: puzzle + random-walk generator (no knobs here) ###
# Vendored (trimmed) from our project's megaminx/src/megaminx/puzzle.py and
# src/cayley/data.py so the notebook is self-contained.
import csv
import json
from dataclasses import dataclass
from typing import Iterable, Sequence

STATE_SIZE = 120
N_GENERATORS = 24
MOVE_SEPARATOR = '.'


@dataclass(frozen=True)
class Megaminx:
    """State = length-120 permutation; solved = identity. 24 generators (12 faces x CW/CCW).

    Convention (matches the competition data and CayleyPy):
        apply(state, gen) -> new_state  where  new_state[i] = state[gen[i]]
    """
    solved_state: tuple
    generators: dict
    move_names: tuple

    @classmethod
    def load(cls, path):
        with open(path, encoding='utf-8') as f:
            info = json.load(f)
        solved = tuple(info['central_state'])
        gens = {name: tuple(perm) for name, perm in info['generators'].items()}
        assert len(solved) == STATE_SIZE and len(gens) == N_GENERATORS
        return cls(solved_state=solved, generators=gens, move_names=tuple(gens.keys()))

    def inverse_name(self, name: str) -> str:
        return name[1:] if name.startswith('-') else '-' + name

    def apply_move(self, state: Sequence[int], move_name: str) -> tuple:
        gen = self.generators[move_name]
        return tuple(state[g] for g in gen)

    def apply_path(self, state: Sequence[int], path: Iterable[str]) -> tuple:
        cur = tuple(state)
        for m in path:
            cur = self.apply_move(cur, m)
        return cur

    def is_solved(self, state: Sequence[int]) -> bool:
        return tuple(state) == self.solved_state

    def parse_path(self, path_str: str) -> list:
        return path_str.split(MOVE_SEPARATOR) if path_str.strip() else []


@dataclass
class GeneratorTable:
    """All generators as one (n_gen, state_size) int64 array + inverse index map."""
    perms: np.ndarray
    names: tuple
    inverse_idx: np.ndarray

    @classmethod
    def from_puzzle(cls, puzzle) -> 'GeneratorTable':
        names = puzzle.move_names
        perms = np.stack([np.array(puzzle.generators[n], dtype=np.int64) for n in names])
        name_to_idx = {n: i for i, n in enumerate(names)}
        inverse_idx = np.array([name_to_idx[puzzle.inverse_name(n)] for n in names],
                               dtype=np.int64)
        return cls(perms=perms, names=names, inverse_idx=inverse_idx)


def generate_walks_torch(puzzle, n_walks, k_max, seed=0, device='cuda', n_back=1,
                         max_resamples=6):
    """Non-backtracking random walks from solved, all on GPU.

    Emits (state_after_step_i, i) for i = 1..k_max per walk. The depth i is an UPPER
    BOUND on the true distance - the model learns a smoothed version that still guides
    beam search. `n_back` bans the inverses of the last n_back actions.

    Returns: states (n_walks*k_max, state_size) int64, depths (n_walks*k_max,) int64.
    """
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    gens = GeneratorTable.from_puzzle(puzzle)
    perms = torch.from_numpy(gens.perms).to(device)
    inv = torch.from_numpy(gens.inverse_idx).to(device)
    n_gen, state_size = perms.shape

    solved = torch.tensor(puzzle.solved_state, dtype=torch.int64, device=device)
    states = solved.unsqueeze(0).expand(n_walks, state_size).clone()
    history = torch.full((n_walks, n_back), -1, dtype=torch.int64, device=device)

    out_states = torch.empty((n_walks * k_max, state_size), dtype=torch.int64, device=device)
    out_depths = torch.empty(n_walks * k_max, dtype=torch.int64, device=device)

    for k in range(1, k_max + 1):
        action = torch.randint(0, n_gen, (n_walks,), generator=g, device=device)
        if n_back > 0:
            hist_safe = torch.where(history >= 0, history, torch.zeros_like(history))
            banned = torch.where(history >= 0, inv[hist_safe], torch.full_like(hist_safe, -1))
            for _ in range(max_resamples):
                bad = (action.unsqueeze(1) == banned).any(dim=1)
                if not bool(bad.any()):
                    break
                n_bad = int(bad.sum().item())
                action = action.clone()
                action[bad] = torch.randint(0, n_gen, (n_bad,), generator=g, device=device)
        states = torch.gather(states, 1, perms[action])
        if n_back > 0:
            history = torch.cat([history[:, 1:], action.unsqueeze(1)], dim=1)
        off = (k - 1) * n_walks
        out_states[off:off + n_walks] = states
        out_depths[off:off + n_walks] = k
    return out_states, out_depths


def apply_all_generators(states, generators):
    """(B, S) x (n_gen, S) -> (B, n_gen, S): children[i, g] = apply(states[i], g)."""
    B, S = states.shape
    n_gen = generators.shape[0]
    return torch.gather(states.unsqueeze(1).expand(B, n_gen, S), 2,
                        generators.unsqueeze(0).expand(B, n_gen, S))


PUZZLE = Megaminx.load(PUZZLE_JSON)
GENS = GeneratorTable.from_puzzle(PUZZLE)
GENERATORS_T = torch.from_numpy(GENS.perms).to(DEVICE)
SOLVED_T = torch.tensor(PUZZLE.solved_state, dtype=torch.int64, device=DEVICE)
with open(TEST_CSV, encoding='utf-8') as f:
    TEST_ROWS = list(csv.DictReader(f))
print(f'puzzle: state_size={STATE_SIZE}, generators={N_GENERATORS}, '
      f'test puzzles={len(TEST_ROWS)}')

In [ ]:
### Library code: MODEL POOL + checkpoint helpers (no knobs here) ###
# Two model families are registered out of the box, selected by CFG['model']['model_type']:
#   'ResMLPDistance'  - our architecture (embedding + LayerNorm ResMLP; AZ v4 = the 6M default)
#   'PilgrimAttnRes'  - the community baseline family from @ogurtsov's modular notebook
#                       (one-hot + BatchNorm; block_type 'residual' or 'attn_res')
# Every registered model works in EVERY stage: stages 1-4 train it as a plain V model
# (native head), stage 5 wraps the same backbone with policy+value heads (DualHeadModel).
#
# To add your own model: define a class with
#     forward(x) -> (B,) [or (B, output_dim)],  features(x) -> (B, feature_dim),
#     a feature_dim attribute, and an output head whose parameter name you register.
# Then:  MODEL_REGISTRY['MyNet'] = {'cls': MyNet, 'head_attr': 'my_head'}
# and put its constructor kwargs under CFG['model']['MyNet'].
import torch.nn as nn
import torch.nn.functional as F


# ---------------- our family: embedding + LayerNorm ResMLP ----------------

class ResBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.lin1 = nn.Linear(dim, dim)
        self.ln1 = nn.LayerNorm(dim)
        self.lin2 = nn.Linear(dim, dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = F.relu(self.ln1(self.lin1(x)))
        h = self.ln2(self.lin2(h))
        return F.relu(x + h)


class ResMLPDistance(nn.Module):
    """Distance predictor: int state (B, 120) -> (B,) scalar (or (B, output_dim)).

    `inference_chunk_size` bounds eval-mode VRAM when beam search hands us 100k+ states.
    """

    def __init__(self, state_size=120, num_classes=120, hidden_dims=(2048, 512),
                 num_res_blocks=2, inference_chunk_size=4096, encoding='embedding',
                 embed_dim=16, output_dim=1):
        super().__init__()
        self.state_size = state_size
        self.num_classes = num_classes
        self.inference_chunk_size = inference_chunk_size
        self.encoding = encoding
        self.embed_dim = embed_dim
        self.output_dim = output_dim
        if encoding == 'onehot':
            in_dim = state_size * num_classes
            self.embedding = None
        elif encoding == 'embedding':
            in_dim = state_size * embed_dim
            self.embedding = nn.Embedding(num_classes, embed_dim)
        else:
            raise ValueError(f'unsupported encoding {encoding!r}')
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.LayerNorm(h), nn.ReLU(inplace=True)]
            prev = h
        self.input_stack = nn.Sequential(*layers)
        self.res_blocks = nn.ModuleList([ResBlock(prev) for _ in range(num_res_blocks)])
        self.head = nn.Linear(prev, output_dim)
        self.feature_dim = prev

    def encode(self, x):
        target_dtype = self.input_stack[0].weight.dtype
        if self.encoding == 'onehot':
            return F.one_hot(x.long(), self.num_classes).to(target_dtype).flatten(start_dim=-2)
        return self.embedding(x.long()).to(target_dtype).flatten(start_dim=-2)

    def features(self, x):
        h = self.encode(x)
        h = self.input_stack(h)
        for block in self.res_blocks:
            h = block(h)
        return h

    def _forward_single(self, x):
        out = self.head(self.features(x))
        return out.squeeze(-1) if self.output_dim == 1 else out

    def forward(self, x):
        if self.training or self.inference_chunk_size is None or x.shape[0] <= self.inference_chunk_size:
            return self._forward_single(x)
        outs = []
        for i in range(0, x.shape[0], self.inference_chunk_size):
            outs.append(self._forward_single(x[i:i + self.inference_chunk_size]))
        return torch.cat(outs, dim=0)

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ------- community family: one-hot + BatchNorm MLP (from @ogurtsov's notebook) -------

class RMSNorm(nn.Module):
    def __init__(self, hidden_dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class BNResidualBlock(nn.Module):
    """Two-layer residual block with BatchNorm (their 'residual' block type)."""

    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        return self.relu(out + x)


class AttnResidualBlock(nn.Module):
    """Residual MLP block with attention over previous block outputs (their 'attn_res')."""

    def __init__(self, hidden_dim, dropout_rate=0.1):
        super().__init__()
        self.fc1 = nn.Linear(hidden_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.query = nn.Parameter(torch.randn(hidden_dim) * 0.02)
        self.rmsnorm = RMSNorm(hidden_dim)
        nn.init.zeros_(self.query)

    def forward(self, x, prev_outputs):
        out = self.fc1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.bn2(out)
        if len(prev_outputs) > 0:
            values = torch.stack(prev_outputs, dim=0)          # (p, B, d)
            keys = self.rmsnorm(values)
            q = self.query.view(1, 1, -1).expand(1, values.size(1), -1)
            logits = torch.einsum('b d, p b d -> p b', q.squeeze(0), keys)
            weights = F.softmax(logits, dim=0).unsqueeze(-1)
            agg = (weights * values).sum(dim=0)
        else:
            agg = torch.zeros_like(out)
        return self.relu(out + agg)


class PilgrimAttnRes(nn.Module):
    """One-hot + BatchNorm MLP with plain or attention-residual blocks.

    Port of @ogurtsov's PilgrimAttnRes, re-parameterized with explicit kwargs and
    extended with `output_dim`, `features()`, and chunked eval-mode forwards so it can
    drive every stage of this pipeline. Math is unchanged.
    """

    def __init__(self, state_size=120, num_classes=120, hd1=1024, hd2=256, nrd=4,
                 n_attn_blocks=0, block_type='residual', dropout_rate=0.0,
                 output_dim=1, inference_chunk_size=4096):
        super().__init__()
        self.state_size = state_size
        self.num_classes = num_classes
        self.block_type = block_type
        self.output_dim = output_dim
        self.inference_chunk_size = inference_chunk_size

        self.input_layer = nn.Linear(state_size * num_classes, hd1)
        self.bn1 = nn.BatchNorm1d(hd1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)
        if hd2 > 0:
            self.hidden_layer = nn.Linear(hd1, hd2)
            self.bn2 = nn.BatchNorm1d(hd2)
            hidden_dim_for_output = hd2
        else:
            self.hidden_layer = None
            self.bn2 = None
            hidden_dim_for_output = hd1
        if block_type == 'residual':
            self.residual_blocks = (nn.ModuleList(
                [BNResidualBlock(hd2, dropout_rate) for _ in range(nrd)])
                if nrd > 0 and hd2 > 0 else None)
        elif block_type == 'attn_res':
            self.residual_blocks = (nn.ModuleList(
                [AttnResidualBlock(hd2, dropout_rate) for _ in range(n_attn_blocks)])
                if n_attn_blocks > 0 and hd2 > 0 else None)
        else:
            raise ValueError(f'unknown block_type {block_type!r}')
        self.output_layer = nn.Linear(hidden_dim_for_output, output_dim)
        self.feature_dim = hidden_dim_for_output

    def features(self, x):
        target_dtype = self.input_layer.weight.dtype
        h = F.one_hot(x.long(), num_classes=self.num_classes).view(x.size(0), -1).to(target_dtype)
        h = self.dropout(self.relu(self.bn1(self.input_layer(h))))
        if self.hidden_layer is not None:
            h = self.dropout(self.relu(self.bn2(self.hidden_layer(h))))
        if self.residual_blocks is not None:
            if self.block_type == 'residual':
                for block in self.residual_blocks:
                    h = block(h)
            else:
                prev_outputs = []
                for block in self.residual_blocks:
                    h = block(h, prev_outputs)
                    prev_outputs.append(h)
        return h

    def _forward_single(self, x):
        out = self.output_layer(self.features(x))
        return out.squeeze(-1) if self.output_dim == 1 else out

    def forward(self, x):
        if self.training or self.inference_chunk_size is None or x.shape[0] <= self.inference_chunk_size:
            return self._forward_single(x)
        outs = []
        for i in range(0, x.shape[0], self.inference_chunk_size):
            outs.append(self._forward_single(x[i:i + self.inference_chunk_size]))
        return torch.cat(outs, dim=0)

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ---------------- registry + builders ----------------

MODEL_REGISTRY = {
    'ResMLPDistance': {'cls': ResMLPDistance, 'head_attr': 'head'},
    'PilgrimAttnRes': {'cls': PilgrimAttnRes, 'head_attr': 'output_layer'},
}


def model_params(model_type=None):
    t = model_type or CFG['model']['model_type']
    return dict(CFG['model'][t])


def build_v_model(model_type=None, params=None, output_dim=1):
    t = model_type or CFG['model']['model_type']
    p = dict(params) if params is not None else model_params(t)
    return MODEL_REGISTRY[t]['cls'](output_dim=output_dim, **p)


def v_model_config(model_type=None, params=None, output_dim=1):
    """Checkpoint model_config. For ResMLPDistance also writes the legacy flat keys so
    exported value heads stay drop-in compatible with our production solver stack."""
    t = model_type or CFG['model']['model_type']
    p = dict(params) if params is not None else model_params(t)
    mc = {'model_type': t, 'params': p, 'output_dim': output_dim}
    if t == 'ResMLPDistance':
        mc.update({'model_class': 'ResMLPDistance', 'state_size': p.get('state_size', 120),
                   'num_classes': p.get('num_classes', 120),
                   'hidden_dims': list(p['hidden_dims']),
                   'num_res_blocks': p['num_res_blocks'],
                   'encoding': 'embedding', 'embed_dim': p['embed_dim']})
    return mc


class DualHeadModel(nn.Module):
    """Any registered backbone + policy head (n_actions) + value head (1).

    The backbone's own native head is carried but unused; policy/value read
    backbone.features(x). This is what the 'az' stage trains.
    """

    def __init__(self, backbone, n_actions):
        super().__init__()
        self.backbone = backbone
        self.n_actions = n_actions
        self.policy_head = nn.Linear(backbone.feature_dim, n_actions)
        self.value_head = nn.Linear(backbone.feature_dim, 1)

    def features(self, x):
        return self.backbone.features(x)

    def forward(self, x):
        h = self.features(x)
        return self.policy_head(h), self.value_head(h).squeeze(-1)

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ---------------- checkpoint helpers ----------------

def clean_state_dict(sd):
    return {k.removeprefix('_orig_mod.'): v for k, v in sd.items()}


def _legacy_resmlp_kwargs(mc):
    keep = ('state_size', 'num_classes', 'num_res_blocks', 'encoding', 'embed_dim',
            'output_dim')
    kw = {k: mc[k] for k in keep if k in mc}
    if 'hidden_dims' in mc:
        kw['hidden_dims'] = tuple(mc['hidden_dims'])
    return kw


def load_v_model(path, device=DEVICE):
    """Load any V checkpoint: new-style (model_type + params) or legacy ResMLP."""
    ck = torch.load(path, map_location='cpu', weights_only=False)
    sd = clean_state_dict(ck['state_dict'])
    mc = ck.get('model_config', {})
    if 'model_type' in mc and 'params' in mc:
        model = build_v_model(mc['model_type'], mc['params'],
                              output_dim=mc.get('output_dim', 1))
    else:
        model = ResMLPDistance(**_legacy_resmlp_kwargs(mc))
    model.load_state_dict(sd)
    model.to(device).eval()
    return model


def warmstart_v_into_dual(dual, ckpt_path, model_type):
    """Copy a V checkpoint into the dual-head backbone; init value_head from its head."""
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = clean_state_dict(ck.get('state_dict', ck))
    backbone_sd = dual.backbone.state_dict()
    compatible = {k: v for k, v in sd.items()
                  if k in backbone_sd and tuple(backbone_sd[k].shape) == tuple(v.shape)}
    skipped = len(sd) - len(compatible)
    dual.backbone.load_state_dict(compatible, strict=False)
    print(f'  warm-start: loaded {len(compatible)} backbone tensors '
          f'({skipped} incompatible skipped)', flush=True)
    if skipped:
        print('  WARNING: incompatible tensors start RANDOM - a different model_type or '
              'shape needs the earlier stages retrained, not a warm-start.', flush=True)
    head = MODEL_REGISTRY[model_type]['head_attr']
    hw, hb = f'{head}.weight', f'{head}.bias'
    if hw in sd and tuple(dual.value_head.weight.shape) == tuple(sd[hw].shape):
        with torch.no_grad():
            dual.value_head.weight.copy_(sd[hw])
            dual.value_head.bias.copy_(sd[hb])
        print('  initialized value_head = V distance head', flush=True)


def export_dual_head(az_ckpt_path, out_path, head='value'):
    """Extract one head of a dual-head checkpoint as a NATIVE V-model checkpoint of the
    same backbone type (value -> beam-search distance model; policy -> n_actions scorer)."""
    ck = torch.load(az_ckpt_path, map_location='cpu', weights_only=False)
    sd = clean_state_dict(ck['state_dict'])
    mc = ck['model_config']
    t, p = mc['model_type'], mc['params']
    out_dim = 1 if head == 'value' else int(mc.get('n_actions', N_GENERATORS))
    head_attr = MODEL_REGISTRY[t]['head_attr']
    new_sd = {k.removeprefix('backbone.'): v for k, v in sd.items()
              if k.startswith('backbone.')}
    src = 'value_head' if head == 'value' else 'policy_head'
    new_sd[f'{head_attr}.weight'] = sd[f'{src}.weight']
    new_sd[f'{head_attr}.bias'] = sd[f'{src}.bias']
    model = build_v_model(t, p, output_dim=out_dim)   # validates shapes on load
    model.load_state_dict(new_sd)
    torch.save({'epoch': ck.get('epoch', -1), 'state_dict': new_sd,
                'model_config': v_model_config(t, p, output_dim=out_dim),
                'source_az_checkpoint': str(az_ckpt_path)}, out_path)
    print(f'exported {head} head -> {out_path}', flush=True)
    return out_path


_probe = build_v_model()
print(f"model pool: {list(MODEL_REGISTRY)} | selected: {CFG['model']['model_type']} "
      f'({_probe.num_parameters():,} params)')
del _probe

In [ ]:
### Policy dataset builder: solutions CSV -> (state, action, remaining) tuples ###
# Every move of every solution contributes one AlphaZero-style training tuple:
# the state before the move, the move taken, and the number of moves remaining.

def build_policy_dataset(submission_csv, out_path):
    move_to_idx = {name: i for i, name in enumerate(PUZZLE.move_names)}
    with open(submission_csv, encoding='utf-8') as f:
        sub_rows = list(csv.DictReader(f))
    all_states, all_actions, all_values = [], [], []
    n_bad = 0
    for row in sub_rows:
        pid = int(row['initial_state_id'])
        path = PUZZLE.parse_path(row['path'])
        if not path:
            continue
        cur = [int(x) for x in TEST_ROWS[pid]['initial_state'].split(',')]
        for i, move in enumerate(path):
            all_states.append(np.array(cur, dtype=np.int8))
            all_actions.append(move_to_idx[move])
            all_values.append(len(path) - i)
            gen = PUZZLE.generators[move]
            cur = [cur[g] for g in gen]
        if tuple(cur) != PUZZLE.solved_state:
            n_bad += 1
    if n_bad:
        print(f'WARNING: {n_bad} paths did not end at solved - check the CSV')
    data = {
        'states': torch.tensor(np.stack(all_states), dtype=torch.int8),
        'actions': torch.tensor(all_actions, dtype=torch.int8),
        'values': torch.tensor(all_values, dtype=torch.float32),
        'source': str(submission_csv),
    }
    torch.save(data, out_path)
    print(f'policy dataset: {len(all_states):,} tuples from {len(sub_rows)} paths '
          f'-> {out_path}', flush=True)
    return out_path


def resolve_policy_dataset():
    """Return the path of the policy dataset the az stage should train on."""
    ds = CFG['az']['policy_dataset']
    if ds not in ('auto', None):
        return Path(ds)
    csv_path = CFG['az']['policy_csv']
    if csv_path in ('auto', None):
        if PREBUILT_POLICY_DATASET.exists():
            return PREBUILT_POLICY_DATASET
        csv_path = DEFAULT_POLICY_CSV
    out = MODELS_DIR / (Path(csv_path).stem + '_az_dataset.pt')
    if not out.exists():
        build_policy_dataset(csv_path, out)
    return out


if 'az' in CFG['stages_to_run']:
    POLICY_DATASET_PATH = resolve_policy_dataset()
    print('policy dataset for the az stage:', POLICY_DATASET_PATH)

In [ ]:
### Stage 1 (pretrain) + stage 2 (curriculum) trainers ###
import copy


def new_v_model():
    return build_v_model(output_dim=1).to(DEVICE)


# Loss factory for the regression stages (same pattern as the community notebook,
# including its two custom losses). Bellman/az losses are structural and stay fixed.
class PinballLoss(nn.Module):
    """Quantile (pinball) loss; tau < 0.5 targets the lower quantile."""

    def __init__(self, tau=0.2):
        super().__init__()
        self.tau = tau

    def forward(self, pred, target):
        e = target - pred
        return torch.mean(torch.maximum(self.tau * e, (self.tau - 1.0) * e))


class LogCoshLoss(nn.Module):
    def forward(self, pred, target):
        ax = torch.abs(pred - target)
        return torch.mean(ax + F.softplus(-2.0 * ax) - 0.6931471805599453)


CUSTOM_LOSSES = {'PinballLoss': PinballLoss, 'LogCoshLoss': LogCoshLoss}


def build_loss(loss_cfg):
    name = (loss_cfg or {}).get('name') or 'MSELoss'
    params = dict((loss_cfg or {}).get('params') or {})
    cls = CUSTOM_LOSSES.get(name) or getattr(nn, name, None)
    if cls is None or not (isinstance(cls, type) and issubclass(cls, nn.Module)):
        raise ValueError(f'unknown loss {name!r} (torch.nn loss or {sorted(CUSTOM_LOSSES)})')
    return cls(**params)


def load_warmstart_strict(model, path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    model.load_state_dict(clean_state_dict(ck['state_dict']))
    print(f'  warm-start (strict) from {Path(path).name}', flush=True)


def save_v_ckpt(path, model, epoch, loss=None, optim=None, sched=None, extra=None):
    ck = {'epoch': epoch, 'state_dict': clean_state_dict(model.state_dict()),
          'model_config': v_model_config(), 'loss': loss}
    if optim is not None:
        ck['optim_state'] = optim.state_dict()
    if sched is not None:
        ck['sched_state'] = sched.state_dict()
    if extra:
        ck.update(extra)
    torch.save(ck, path)


def try_resume(model, optim, sched, resume_path):
    """Returns the epoch to start from (0 if no resume)."""
    if not resume_path:
        return 0
    ck = torch.load(resume_path, map_location='cpu', weights_only=False)
    model.load_state_dict(clean_state_dict(ck['state_dict']))
    if optim is not None and 'optim_state' in ck:
        optim.load_state_dict(ck['optim_state'])
    if sched is not None and 'sched_state' in ck:
        sched.load_state_dict(ck['sched_state'])
    start = int(ck.get('epoch', -1)) + 1
    print(f'  resumed from {resume_path} at epoch {start}', flush=True)
    return start


def iterate_batches(n, batch_size, generator, device):
    idx = torch.randperm(n, generator=generator, device=device)
    return [idx[i:i + batch_size] for i in range(0, n, batch_size)]


def run_pretrain(cfg, warmstart=None):
    """Plain random-walk MSE regression, fresh walks every epoch."""
    out_dir = MODELS_DIR / 'pretrain'
    out_dir.mkdir(parents=True, exist_ok=True)
    model = new_v_model()
    if warmstart:
        load_warmstart_strict(model, warmstart)
    optim = make_optimizer(model.parameters(), cfg)
    loss_fn = build_loss(cfg.get('loss'))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg['epochs'])
    start_epoch = try_resume(model, optim, sched, cfg['resume'])
    model_c = maybe_compile(model)
    autocast_ctx = make_autocast()
    batch_gen = torch.Generator(device=DEVICE)
    batch_gen.manual_seed(CFG['seed'])
    last_path = None
    for epoch in range(start_epoch, cfg['epochs']):
        t0 = time.time()
        n_walks = max(1, cfg['samples_per_epoch'] // cfg['k_max'])
        states, depths = generate_walks_torch(PUZZLE, n_walks=n_walks, k_max=cfg['k_max'],
                                              seed=CFG['seed'] + epoch, device=DEVICE,
                                              n_back=cfg['n_back'])
        depths_f = depths.to(torch.float32)
        model_c.train()
        total, nb = 0.0, 0
        for idx in iterate_batches(states.shape[0], cfg['batch_size'], batch_gen, DEVICE):
            with autocast_ctx:
                pred = model_c(states[idx])
                loss = loss_fn(pred, depths_f[idx])
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            total += float(loss.item())
            nb += 1
        sched.step()
        avg = total / max(nb, 1)
        if epoch % 25 == 0 or epoch == cfg['epochs'] - 1:
            print(f'epoch {epoch:4d} | loss {avg:.4f} | lr {sched.get_last_lr()[0]:.2e} '
                  f'| {time.time() - t0:.1f}s', flush=True)
        if (epoch + 1) % cfg['checkpoint_every'] == 0 or epoch == cfg['epochs'] - 1:
            last_path = out_dir / f'epoch_{epoch:04d}.pt'
            save_v_ckpt(last_path, model, epoch, loss=avg, optim=optim, sched=sched)
        if training_time_up() and epoch < cfg['epochs'] - 1:
            last_path = out_dir / f'epoch_{epoch:04d}.pt'
            save_v_ckpt(last_path, model, epoch, loss=avg, optim=optim, sched=sched)
            print(f'*** wall budget reached at epoch {epoch} - stopping pretrain; '
                  f'the next run resumes from {last_path.name}', flush=True)
            return last_path, False
    return last_path, True


@torch.no_grad()
def _ema_update(ema_model, source, tau):
    for ep_, sp in zip(ema_model.parameters(), source.parameters()):
        ep_.mul_(1.0 - tau).add_(sp.detach(), alpha=tau)
    for eb, sb in zip(ema_model.buffers(), source.buffers()):
        if eb.dtype.is_floating_point:
            eb.mul_(1.0 - tau).add_(sb.detach(), alpha=tau)
        else:
            eb.copy_(sb)


@torch.no_grad()
def _eval_val_mse(model, val_states, val_depths, batch_size=8192):
    model.eval()
    total, seen = 0.0, 0
    for i in range(0, val_states.size(0), batch_size):
        bs = val_states[i:i + batch_size]
        bd = val_depths[i:i + batch_size]
        total += float(F.mse_loss(model(bs), bd, reduction='sum').item())
        seen += bs.size(0)
    return total / max(seen, 1)


def run_curriculum(cfg, warmstart):
    """Mixed-k curriculum fine-tune with per-step EMA and val-based early stopping.

    Output = best_ema.pt (the EMA snapshot at the best validation epoch).
    """
    out_dir = MODELS_DIR / 'curriculum'
    out_dir.mkdir(parents=True, exist_ok=True)
    model = new_v_model()
    if warmstart:  # None when resuming (the resume checkpoint restores everything)
        load_warmstart_strict(model, warmstart)
    ema_model = copy.deepcopy(model).eval()
    for p in ema_model.parameters():
        p.requires_grad = False
    optim = make_optimizer(model.parameters(), cfg)
    loss_fn = build_loss(cfg.get('loss'))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg['budget_epochs'])
    start_epoch = try_resume(model, optim, sched, cfg['resume'])
    if cfg['resume']:
        ck = torch.load(cfg['resume'], map_location='cpu', weights_only=False)
        if 'ema_state_dict' in ck:
            ema_model.load_state_dict(clean_state_dict(ck['ema_state_dict']))
    model_c = maybe_compile(model)
    autocast_ctx = make_autocast()
    base_model = getattr(model_c, '_orig_mod', model_c)
    batch_gen = torch.Generator(device=DEVICE)
    batch_gen.manual_seed(CFG['seed'])
    k_gen = torch.Generator(device=DEVICE)
    k_gen.manual_seed(CFG['seed'] + 7777)

    mix_k = list(cfg['mix_k_list'])
    print(f"curriculum: warmup k={cfg['warmup_k_max']} for {cfg['warmup_epochs']} epochs, "
          f'then mix from {mix_k}', flush=True)
    val_states, val_depths = generate_walks_torch(
        PUZZLE, n_walks=cfg['val_n_walks'], k_max=cfg['val_k_max'],
        seed=CFG['seed'] - 1, device=DEVICE, n_back=cfg['n_back'])
    val_depths = val_depths.to(torch.float32)
    print(f'  fixed val set: {val_states.size(0):,} states', flush=True)

    best_val, best_epoch = float('inf'), -1
    best_path = out_dir / 'best_ema.pt'
    for epoch in range(start_epoch, cfg['budget_epochs']):
        t0 = time.time()
        if epoch < cfg['warmup_epochs']:
            k_for_epoch = cfg['warmup_k_max']
        else:
            k_for_epoch = mix_k[int(torch.randint(0, len(mix_k), (1,), generator=k_gen,
                                                  device=DEVICE).item())]
        n_walks = max(1, cfg['samples_per_epoch'] // k_for_epoch)
        states, depths = generate_walks_torch(PUZZLE, n_walks=n_walks, k_max=k_for_epoch,
                                              seed=CFG['seed'] + epoch, device=DEVICE,
                                              n_back=cfg['n_back'])
        depths_f = depths.to(torch.float32)
        model_c.train()
        total, nb = 0.0, 0
        for idx in iterate_batches(states.shape[0], cfg['batch_size'], batch_gen, DEVICE):
            with autocast_ctx:
                pred = model_c(states[idx])
                loss = loss_fn(pred, depths_f[idx])
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            _ema_update(ema_model, base_model, cfg['ema_tau'])
            total += float(loss.item())
            nb += 1
        sched.step()

        if (epoch + 1) % cfg['val_every'] == 0 or epoch == cfg['budget_epochs'] - 1:
            val_ema = _eval_val_mse(ema_model, val_states, val_depths)
            print(f'epoch {epoch:5d} | k={k_for_epoch:<3} | train_mse {total / max(nb, 1):7.3f} '
                  f'| val_mse ema {val_ema:7.3f} | lr {sched.get_last_lr()[0]:.2e} '
                  f'| {time.time() - t0:.1f}s', flush=True)
            if val_ema < best_val:
                best_val, best_epoch = val_ema, epoch
                save_v_ckpt(best_path, ema_model, epoch, loss=val_ema,
                            extra={'val_mse': val_ema})
                print(f'  -> new best EMA (val_mse={val_ema:.4f}) saved', flush=True)
            elif (epoch - best_epoch >= cfg['patience']
                  and epoch >= cfg['warmup_epochs'] + cfg['patience']):
                print(f'*** early stop at epoch {epoch}: best val_mse {best_val:.4f} '
                      f'at epoch {best_epoch}', flush=True)
                break
        if (epoch + 1) % cfg['checkpoint_every'] == 0:
            save_v_ckpt(out_dir / f'epoch_{epoch:05d}.pt', base_model, epoch,
                        optim=optim, sched=sched,
                        extra={'ema_state_dict': clean_state_dict(ema_model.state_dict())})
        if training_time_up() and epoch < cfg['budget_epochs'] - 1:
            stop_path = out_dir / f'epoch_{epoch:05d}.pt'
            save_v_ckpt(stop_path, base_model, epoch, optim=optim, sched=sched,
                        extra={'ema_state_dict': clean_state_dict(ema_model.state_dict())})
            print(f'*** wall budget reached at epoch {epoch} - stopping curriculum; '
                  f'the next run resumes from {stop_path.name} '
                  f'(note: best-val/patience tracking restarts on resume)', flush=True)
            return stop_path, False
    assert best_path.exists(), 'curriculum produced no best_ema.pt (increase budget?)'
    return best_path, True

In [ ]:
### Stages 3+4 trainer: Bellman bootstrap with frontier / BFS-d6 / anchor mixins ###

_BFS6_CACHE = {}


def load_bfs6():
    """(states int8 (N,120), distances float32 (N,)) on CPU; cached across stages."""
    if 'data' not in _BFS6_CACHE:
        print(f'loading BFS-d6 dataset ({BFS_D6_PATH.name}, ~2.3GB - takes a while '
              f'from the input mount)...', flush=True)
        d = torch.load(BFS_D6_PATH, map_location='cpu', weights_only=False)
        _BFS6_CACHE['data'] = (d['states'], d['distances'].to(torch.float32))
        print(f"  {d['states'].size(0):,} exact-distance states loaded", flush=True)
    return _BFS6_CACHE['data']


def load_frontier():
    if 'frontier' not in _BFS6_CACHE:
        d = torch.load(FRONTIER_PATH, map_location='cpu', weights_only=False)
        _BFS6_CACHE['frontier'] = d['states']
        print(f"  {d['states'].size(0):,} frontier states loaded", flush=True)
    return _BFS6_CACHE['frontier']


@torch.no_grad()
def bellman_targets(forward_fn, states, walk_depths, chunk_size,
                    clip_upper=True, clip_lower=True):
    """target = clip(1 + min_a V_target(apply(s, a)), 0, walk_depth).

    A child equal to the solved state contributes exactly 0 (the boundary condition).
    `forward_fn(chunk) -> (B,) float` is the frozen target net's value forward.
    """
    B, S = states.shape
    children_flat = apply_all_generators(states, GENERATORS_T).reshape(-1, S)
    is_solved = (children_flat == SOLVED_T).all(dim=1)
    vals = torch.empty(children_flat.size(0), dtype=torch.float32, device=states.device)
    for i in range(0, children_flat.size(0), chunk_size):
        vals[i:i + chunk_size] = forward_fn(children_flat[i:i + chunk_size]).flatten().float()
    vals = torch.where(is_solved, torch.zeros_like(vals), vals).view(B, -1)
    target = 1.0 + vals.min(dim=1).values
    if clip_upper:
        target = torch.minimum(target, walk_depths)
    if clip_lower:
        target = torch.clamp(target, min=0.0)
    return target


def run_bellman(cfg, warmstart, stage_name):
    """Bellman refinement. stage_name in ('bellman', 'bellman_dd') - only the anchor
    counts (and epochs) differ between the two stages."""
    out_dir = MODELS_DIR / stage_name
    out_dir.mkdir(parents=True, exist_ok=True)
    model = new_v_model()
    if warmstart:  # None when resuming (the resume checkpoint restores everything)
        load_warmstart_strict(model, warmstart)
    target_model = copy.deepcopy(model).eval()
    for p in target_model.parameters():
        p.requires_grad = False
    optim = make_optimizer(model.parameters(), cfg)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg['epochs'])
    start_epoch = try_resume(model, optim, sched, cfg['resume'])
    if start_epoch > 0:
        target_model.load_state_dict(clean_state_dict(model.state_dict()))
    model_c = maybe_compile(model)
    autocast_ctx = make_autocast()
    batch_gen = torch.Generator(device=DEVICE)
    batch_gen.manual_seed(CFG['seed'])
    cpu_gen = torch.Generator(device='cpu')
    cpu_gen.manual_seed(CFG['seed'] + 12345)

    # Anchors (stage 4): V(solved)=0 and the 24 depth-1 children = 1, exact, every batch.
    anchor_v0 = SOLVED_T.unsqueeze(0)                                    # (1, S)
    anchor_d1 = apply_all_generators(anchor_v0, GENERATORS_T).squeeze(0)  # (24, S)
    if cfg['n_anchor_v0'] or cfg['n_anchor_d1']:
        print(f"[{stage_name}] anchor mixin: V0 x {cfg['n_anchor_v0']}, "
              f"d=1 children 24 x {cfg['n_anchor_d1']}", flush=True)

    bfs6_states = bfs6_dists = None
    bfs6_per_batch = 0
    if cfg['bfs_d6_fraction'] > 0:
        bfs6_states, bfs6_dists = load_bfs6()
        bfs6_per_batch = max(1, int(round(cfg['batch_size'] * cfg['bfs_d6_fraction'])))
    fr_states = None
    fr_per_batch = 0
    if cfg['frontier_fraction'] > 0:
        fr_states = load_frontier()
        fr_per_batch = max(1, int(round(cfg['batch_size'] * cfg['frontier_fraction'])))
    print(f"[{stage_name}] per batch: rw {cfg['batch_size'] - bfs6_per_batch - fr_per_batch}"
          f' + bfs6 {bfs6_per_batch} + frontier {fr_per_batch}'
          f" + anchors {cfg['n_anchor_v0'] + 24 * cfg['n_anchor_d1']}", flush=True)

    target_fwd = lambda x: target_model(x)
    last_path = None
    for epoch in range(start_epoch, cfg['epochs']):
        t0 = time.time()
        n_walks = max(1, cfg['samples_per_epoch'] // cfg['k_max'])
        states, depths = generate_walks_torch(PUZZLE, n_walks=n_walks, k_max=cfg['k_max'],
                                              seed=CFG['seed'] + epoch, device=DEVICE,
                                              n_back=cfg['n_back'])
        depths_f = depths.to(torch.float32)
        rw_per_batch = cfg['batch_size'] - bfs6_per_batch - fr_per_batch

        # Pre-sample this epoch's mixin rows (fresh shuffle per epoch).
        n_batches_est = states.size(0) // rw_per_batch + 1
        if bfs6_states is not None:
            take = min(bfs6_per_batch * n_batches_est, bfs6_states.size(0))
            perm = torch.randperm(bfs6_states.size(0), generator=cpu_gen)[:take]
            ep_bfs_states = bfs6_states[perm].to(DEVICE)
            ep_bfs_dists = bfs6_dists[perm].to(DEVICE)
        if fr_states is not None:
            take = min(fr_per_batch * n_batches_est, fr_states.size(0))
            perm = torch.randperm(fr_states.size(0), generator=cpu_gen)[:take]
            ep_fr_states = fr_states[perm].to(DEVICE)

        model_c.train()
        total, nb = 0.0, 0
        bfs_cur = fr_cur = 0
        for idx in iterate_batches(states.shape[0], rw_per_batch, batch_gen, DEVICE):
            bs_rw = states[idx]
            bd_rw = depths_f[idx]
            target_rw = bellman_targets(target_fwd, bs_rw, bd_rw, cfg['target_net_chunk'],
                                        cfg['clip_upper'], cfg['clip_lower'])
            bs_parts, tg_parts = [bs_rw], [target_rw]
            if cfg['n_anchor_v0'] > 0:
                bs_parts.append(anchor_v0.expand(cfg['n_anchor_v0'], -1))
                tg_parts.append(torch.zeros(cfg['n_anchor_v0'], device=DEVICE))
            if cfg['n_anchor_d1'] > 0:
                bs_parts.append(anchor_d1.repeat(cfg['n_anchor_d1'], 1))
                tg_parts.append(torch.ones(N_GENERATORS * cfg['n_anchor_d1'], device=DEVICE))
            if bfs6_states is not None and bfs_cur + bfs6_per_batch <= ep_bfs_states.size(0):
                bs_parts.append(ep_bfs_states[bfs_cur:bfs_cur + bfs6_per_batch])
                tg_parts.append(ep_bfs_dists[bfs_cur:bfs_cur + bfs6_per_batch])
                bfs_cur += bfs6_per_batch
            if fr_states is not None and fr_cur + fr_per_batch <= ep_fr_states.size(0):
                bs_fr = ep_fr_states[fr_cur:fr_cur + fr_per_batch]
                fr_cur += fr_per_batch
                # Frontier states have no walk-depth bound; use a synthetic cap so
                # clip_upper effectively no-ops while clip_lower still applies.
                bd_fr = torch.full((bs_fr.size(0),), float(cfg['frontier_walk_depth_cap']),
                                   device=DEVICE)
                tg_fr = bellman_targets(target_fwd, bs_fr, bd_fr, cfg['target_net_chunk'],
                                        cfg['clip_upper'], cfg['clip_lower'])
                bs_parts.append(bs_fr)
                tg_parts.append(tg_fr)
            bs = torch.cat([p.to(torch.int64) for p in bs_parts], dim=0)
            tg = torch.cat(tg_parts, dim=0)
            with autocast_ctx:
                pred = model_c(bs)
                loss = F.mse_loss(pred.float(), tg)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            total += float(loss.item())
            nb += 1
        sched.step()
        avg = total / max(nb, 1)
        if epoch % 5 == 0 or epoch == cfg['epochs'] - 1:
            print(f'epoch {epoch:4d} | loss {avg:.4f} | lr {sched.get_last_lr()[0]:.2e} '
                  f'| {time.time() - t0:.1f}s', flush=True)
        if (epoch + 1) % cfg['target_update_every'] == 0:
            target_model.load_state_dict(clean_state_dict(model.state_dict()))
        if (epoch + 1) % cfg['checkpoint_every'] == 0 or epoch == cfg['epochs'] - 1:
            last_path = out_dir / f'epoch_{epoch:04d}.pt'
            save_v_ckpt(last_path, model, epoch, loss=avg, optim=optim, sched=sched)
        if training_time_up() and epoch < cfg['epochs'] - 1:
            last_path = out_dir / f'epoch_{epoch:04d}.pt'
            save_v_ckpt(last_path, model, epoch, loss=avg, optim=optim, sched=sched)
            print(f'*** wall budget reached at epoch {epoch} - stopping {stage_name}; '
                  f'the next run resumes from {last_path.name}', flush=True)
            return last_path, False
    return last_path, True

In [ ]:
### Stage 5 trainer: dual-head AZ fine-tune (policy CE + Bellman value) ###

@torch.no_grad()
def az_value_forward(model, states, chunk_size=4096):
    outs = []
    for i in range(0, states.size(0), chunk_size):
        h = model.features(states[i:i + chunk_size])
        outs.append(model.value_head(h).squeeze(-1))
    return torch.cat(outs, dim=0)


@torch.no_grad()
def _quick_value_canary(model):
    """V(solved) and mean V(depth-1) from the value head - printed at checkpoints."""
    model.eval()  # BatchNorm backbones cannot do a batch-of-1 forward in train mode
    v0 = az_value_forward(model, SOLVED_T.unsqueeze(0)).item()
    d1 = apply_all_generators(SOLVED_T.unsqueeze(0), GENERATORS_T).squeeze(0)
    v1 = az_value_forward(model, d1).mean().item()
    return v0, v1


def run_az(cfg, warmstart):
    """Reproduces the AZ v4 run: value head trained like stage 4 (minus frontier),
    policy head CE on solution paths, shared trunk. STOP EARLY - see select_epoch."""
    out_dir = MODELS_DIR / 'az'
    out_dir.mkdir(parents=True, exist_ok=True)
    model_type = CFG['model']['model_type']
    model = DualHeadModel(build_v_model(output_dim=1),
                          n_actions=N_GENERATORS).to(DEVICE)
    print(f'AZ model: {model_type} backbone, {model.num_parameters():,} params', flush=True)
    if warmstart:  # None when resuming (the resume checkpoint restores everything)
        warmstart_v_into_dual(model, warmstart, model_type)
    target_model = copy.deepcopy(model).eval()
    for p in target_model.parameters():
        p.requires_grad = False

    policy = torch.load(POLICY_DATASET_PATH, map_location='cpu', weights_only=False)
    states_p = policy['states'].to(DEVICE).long()
    actions_p = policy['actions'].to(DEVICE).long()
    Np = states_p.size(0)
    print(f'policy dataset: {Np:,} (state, action) pairs '
          f"from {policy.get('source', '?')}", flush=True)

    bfs6_states = bfs6_dists = None
    bfs6_per_batch = 0
    if cfg['bfs_d6_fraction'] > 0:
        bfs6_states, bfs6_dists = load_bfs6()
        bfs6_per_batch = max(1, int(round(cfg['rw_batch_size'] * cfg['bfs_d6_fraction'])))

    optim = make_optimizer(model.parameters(), cfg)
    # NOTE: cosine T_max is lr_t_max (=200), NOT epochs - reproduces the original
    # schedule even when running only the first ~30 epochs.
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=cfg['lr_t_max'])
    start_epoch = try_resume(model, optim, sched, cfg['resume'])
    if start_epoch > 0:
        target_model.load_state_dict(clean_state_dict(model.state_dict()))
    autocast_ctx = make_autocast()
    rng_p = torch.Generator(device=DEVICE)
    rng_p.manual_seed(CFG['seed'])
    cpu_gen = torch.Generator(device='cpu')
    cpu_gen.manual_seed(CFG['seed'] + 1)

    anchor_v0 = SOLVED_T.unsqueeze(0)
    anchor_d1 = apply_all_generators(anchor_v0, GENERATORS_T).squeeze(0)
    target_fwd = lambda x: az_value_forward(target_model, x)

    print(f"epochs: {cfg['epochs']}  rw_batch: {cfg['rw_batch_size']}  "
          f"policy_batch: {cfg['policy_batch_size']}  alpha={cfg['alpha']}  "
          f"beta={cfg['beta']}", flush=True)

    def save_az_ckpt(epoch, avg_p, avg_v, acc):
        path = out_dir / f'epoch_{epoch:04d}.pt'
        torch.save({
            'epoch': epoch, 'state_dict': clean_state_dict(model.state_dict()),
            'model_config': {'model_type': model_type, 'params': model_params(),
                             'n_actions': N_GENERATORS},
            'p_loss': avg_p, 'v_loss': avg_v, 'top1_acc': acc,
            'optim_state': optim.state_dict(), 'sched_state': sched.state_dict(),
        }, path)
        return path

    ckpts = {}
    completed = True
    for epoch in range(start_epoch, cfg['epochs']):
        t0 = time.time()
        n_walks = max(1, cfg['samples_per_epoch'] // cfg['k_max'])
        rw_states, rw_depths = generate_walks_torch(
            PUZZLE, n_walks=n_walks, k_max=cfg['k_max'],
            seed=CFG['seed'] + epoch * 1000, device=DEVICE, n_back=cfg['n_back'])
        rw_depths_f = rw_depths.to(torch.float32)
        N_rw = rw_states.size(0)

        rw_per_batch = (cfg['rw_batch_size'] - bfs6_per_batch
                        - cfg['anchor_v0'] - N_GENERATORS * cfg['anchor_d1'])
        if bfs6_states is not None:
            take = min(bfs6_per_batch * (N_rw // rw_per_batch + 1), bfs6_states.size(0))
            perm = torch.randperm(bfs6_states.size(0), generator=cpu_gen)[:take]
            ep_bfs_states = bfs6_states[perm].to(DEVICE).long()
            ep_bfs_dists = bfs6_dists[perm].to(DEVICE)

        n_rw_batches = max(1, N_rw // rw_per_batch)
        bfs_cur = 0
        tot_p = tot_v = tot_acc = 0.0
        model.train()
        for b in range(n_rw_batches):
            bs_rw = rw_states[b * rw_per_batch:(b + 1) * rw_per_batch]
            bd_rw = rw_depths_f[b * rw_per_batch:(b + 1) * rw_per_batch]
            target_rw = bellman_targets(target_fwd, bs_rw, bd_rw, chunk_size=4096)

            bs_parts, tg_parts = [bs_rw], [target_rw]
            if cfg['anchor_v0'] > 0:
                bs_parts.append(anchor_v0.expand(cfg['anchor_v0'], -1))
                tg_parts.append(torch.zeros(cfg['anchor_v0'], device=DEVICE))
            if cfg['anchor_d1'] > 0:
                bs_parts.append(anchor_d1.repeat(cfg['anchor_d1'], 1))
                tg_parts.append(torch.ones(N_GENERATORS * cfg['anchor_d1'], device=DEVICE))
            if bfs6_states is not None and bfs_cur + bfs6_per_batch <= ep_bfs_states.size(0):
                bs_parts.append(ep_bfs_states[bfs_cur:bfs_cur + bfs6_per_batch])
                tg_parts.append(ep_bfs_dists[bfs_cur:bfs_cur + bfs6_per_batch])
                bfs_cur += bfs6_per_batch
            bs_v = torch.cat(bs_parts, dim=0)
            tg_v = torch.cat(tg_parts, dim=0)

            idx_p = torch.randint(0, Np, (cfg['policy_batch_size'],), generator=rng_p,
                                  device=DEVICE)
            bs_p = states_p[idx_p]
            ba_p = actions_p[idx_p]

            with autocast_ctx:
                h_v = model.features(bs_v)
                pred_v = model.value_head(h_v).squeeze(-1)
                v_loss = F.mse_loss(pred_v.float(), tg_v)
                h_p = model.features(bs_p)
                logits_p = model.policy_head(h_p)
                p_loss = F.cross_entropy(logits_p, ba_p)
                loss = cfg['alpha'] * p_loss + cfg['beta'] * v_loss
            optim.zero_grad(set_to_none=True)
            loss.backward()
            optim.step()
            tot_p += float(p_loss.item())
            tot_v += float(v_loss.item())
            with torch.no_grad():
                tot_acc += float((logits_p.argmax(dim=1) == ba_p).float().mean().item())
        sched.step()

        avg_p, avg_v = tot_p / n_rw_batches, tot_v / n_rw_batches
        acc = tot_acc / n_rw_batches
        print(f'epoch {epoch:4d} | p_loss {avg_p:.4f} | v_loss {avg_v:.4f} | '
              f'top-1 acc {acc:.4f} | lr {sched.get_last_lr()[0]:.2e} '
              f'| {time.time() - t0:.1f}s', flush=True)
        if acc > 0.40:
            print('  NOTE: top-1 acc > 40% - likely PAST the value-quality sweet spot; '
                  'prefer an earlier checkpoint.', flush=True)

        if (epoch + 1) % cfg['target_update_every'] == 0:
            target_model.load_state_dict(model.state_dict())

        if (epoch + 1) % cfg['checkpoint_every'] == 0 or epoch == cfg['epochs'] - 1:
            path = save_az_ckpt(epoch, avg_p, avg_v, acc)
            ckpts[epoch] = path
            v0, v1 = _quick_value_canary(model)
            model.train()
            print(f'  checkpoint {path.name} | V(solved)={v0:+.3f} V(d=1)={v1:+.3f} '
                  f'(want ~0 and ~1)', flush=True)
        if training_time_up() and epoch < cfg['epochs'] - 1:
            ckpts[epoch] = save_az_ckpt(epoch, avg_p, avg_v, acc)
            print(f'*** wall budget reached at epoch {epoch} - stopping az; '
                  f'the next run resumes from {ckpts[epoch].name}', flush=True)
            completed = False
            break

    sel = cfg['select_epoch']
    if sel == 'last' or sel not in ckpts:
        sel_epoch = max(ckpts)
        if sel != 'last' and completed:
            print(f'select_epoch {sel} was not checkpointed; using last ({sel_epoch})')
    else:
        sel_epoch = sel
    if not completed:
        sel_epoch = max(ckpts)  # incomplete stage: the resume point, not a selection
    print(f'selected AZ checkpoint: epoch {sel_epoch}', flush=True)
    return ckpts[sel_epoch], completed

In [ ]:
### Run the selected stages (in pipeline order, chaining warm-starts) ###
PREV_STAGE = {'pretrain': None, 'curriculum': 'pretrain', 'bellman': 'curriculum',
              'bellman_dd': 'bellman', 'az': 'bellman_dd'}
STAGE_OUTPUTS = {}

# The shipped warm-start checkpoints are for the DEFAULT model only.
_DEFAULT_MODEL = ('ResMLPDistance',
                  {'hidden_dims': (2048, 512), 'num_res_blocks': 2, 'embed_dim': 16})


def model_is_default():
    t = CFG['model']['model_type']
    if t != _DEFAULT_MODEL[0]:
        return False
    p = model_params()
    ref = _DEFAULT_MODEL[1]
    return (tuple(p.get('hidden_dims', ())) == ref['hidden_dims']
            and p.get('num_res_blocks') == ref['num_res_blocks']
            and p.get('embed_dim') == ref['embed_dim'])


def resolve_warmstart(stage):
    w = CFG[stage].get('warmstart')
    if w not in ('auto', None):
        return Path(w)
    if w is None:
        return None
    prev = PREV_STAGE[stage]
    if prev in STAGE_OUTPUTS:
        print(f'[{stage}] warm-start from freshly trained {prev}: {STAGE_OUTPUTS[prev]}')
        return STAGE_OUTPUTS[prev]
    if not model_is_default():
        raise AssertionError(
            f"[{stage}] the shipped warm-start checkpoints are for the default 6M "
            f"ResMLPDistance, but model_type={CFG['model']['model_type']} with "
            f'params={model_params()}. Either include the earlier stages in '
            f"stages_to_run (train this model from 'pretrain' up), or set an explicit "
            f"'warmstart' path for this stage.")
    asset = ASSETS / STAGE_INPUT_ASSET[stage]
    assert asset.exists(), (
        f'{stage}: no fresh {prev} output and shipped warm-start missing: {asset}')
    print(f'[{stage}] warm-start from shipped asset: {asset.name}')
    return asset


# ---- multi-session chain state (models/chain_state.json in the run output) ----
CHAIN_STATE_PATH = MODELS_DIR / 'chain_state.json'
_FINGERPRINT = json.loads(json.dumps(
    {'experiment': CFG.get('experiment', 'az4-baseline'),
     'model_type': CFG['model']['model_type'], 'params': model_params(),
     'seed': CFG['seed']}, default=list))


def _rel_models(p):
    parts = Path(p).parts
    return Path(*parts[parts.index('models'):]).as_posix()


def _write_chain_state(completed, partial=None):
    state = {'fingerprint': _FINGERPRINT,
             'completed_rel': {s: _rel_models(p) for s, p in completed.items()},
             'partial_rel': ({'stage': partial['stage'],
                              'resume': _rel_models(partial['resume'])}
                             if partial else None)}
    CHAIN_STATE_PATH.write_text(json.dumps(state, indent=2), encoding='utf-8')


def _load_prev_chain_state():
    if not CFG.get('auto_resume', True) or PREV_OUTPUT is None:
        return {}, None
    try:
        state = json.loads((PREV_OUTPUT / 'models' / 'chain_state.json')
                           .read_text(encoding='utf-8'))
    except Exception as e:
        print('could not read previous chain_state:', e)
        return {}, None
    prev_fp = dict(state.get('fingerprint') or {})
    # chains written before the 'experiment' field existed belong to the default label
    prev_fp.setdefault('experiment', 'az4-baseline')
    if prev_fp != _FINGERPRINT:
        print('previous run was a different experiment/model/seed - ignoring its progress')
        print('  previous:', prev_fp)
        print('  current :', _FINGERPRINT)
        return {}, None
    completed = {s: str(PREV_OUTPUT / rel)
                 for s, rel in (state.get('completed_rel') or {}).items()}
    partial = state.get('partial_rel')
    if partial:
        partial = {'stage': partial['stage'],
                   'resume': str(PREV_OUTPUT / partial['resume'])}
    return completed, partial


completed_paths, prev_partial = _load_prev_chain_state()
if completed_paths:
    print('continuing chain: previously completed stages:', sorted(completed_paths))
if prev_partial:
    print(f"continuing chain: '{prev_partial['stage']}' resumes from "
          f"{prev_partial['resume']}")

# Kaggle mounts only the LATEST successful run's output, and each run's output only
# contains what that run wrote - so copy the chain's live artifacts forward into THIS
# run's output, making every successful output self-contained. Entries whose file is
# missing (outputs produced before this fix) are kept as stale pointers with a warning;
# they only matter if a later stage tries to load them.
import shutil


def _pull_forward(path_str, what):
    src = Path(path_str)
    dst = WORK / _rel_models(src)
    if src == dst or dst.exists():
        return str(dst) if dst.exists() else path_str
    if not src.exists():
        print(f'WARNING: {what} checkpoint missing from previous output ({src}) - '
              f'keeping a stale pointer; only a problem if something loads it.')
        return path_str
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return str(dst)


for _s in list(completed_paths):
    completed_paths[_s] = _pull_forward(completed_paths[_s], f'completed {_s}')
if prev_partial:
    prev_partial['resume'] = _pull_forward(prev_partial['resume'],
                                           f"partial {prev_partial['stage']}")

chain_stopped = False
current_partial = None

for stage in STAGE_ORDER:
    if stage not in CFG['stages_to_run']:
        continue
    if stage in completed_paths:
        STAGE_OUTPUTS[stage] = Path(completed_paths[stage])
        print(f'[{stage}] already completed in a previous run -> {STAGE_OUTPUTS[stage]}')
        continue
    if training_time_up():
        print(f"[{stage}] wall budget exhausted before this stage - re-run the "
              f'notebook to continue the chain.', flush=True)
        chain_stopped = True
        current_partial = prev_partial if (prev_partial
                                           and prev_partial['stage'] == stage) else None
        break
    if (prev_partial and prev_partial['stage'] == stage
            and CFG[stage].get('resume') is None):
        CFG[stage]['resume'] = prev_partial['resume']
    # A resuming stage restores its full weights from the resume checkpoint - the
    # warm-start would be loaded and immediately overwritten, so skip it (this also
    # avoids depending on the previous stage's file at all when resuming).
    resuming = CFG[stage].get('resume') is not None
    print('\n' + '=' * 70)
    print(f'STAGE: {stage}')
    print('resolved config:', json.dumps(CFG[stage], default=str))  # echo before running
    print('=' * 70, flush=True)
    t_stage = time.time()
    if stage == 'pretrain':
        out, done = run_pretrain(CFG['pretrain'], warmstart=None)
    elif stage == 'curriculum':
        out, done = run_curriculum(CFG['curriculum'],
                                   None if resuming else resolve_warmstart('curriculum'))
    elif stage == 'bellman':
        out, done = run_bellman(CFG['bellman'],
                                None if resuming else resolve_warmstart('bellman'),
                                'bellman')
    elif stage == 'bellman_dd':
        out, done = run_bellman(CFG['bellman_dd'],
                                None if resuming else resolve_warmstart('bellman_dd'),
                                'bellman_dd')
    elif stage == 'az':
        out, done = run_az(CFG['az'], None if resuming else resolve_warmstart('az'))
    print(f'[{stage}] {"done" if done else "PAUSED"} in '
          f'{(time.time() - t_stage) / 60:.1f} min -> {out}', flush=True)
    if done:
        STAGE_OUTPUTS[stage] = out
        completed_paths[stage] = str(out)
        _write_chain_state(completed_paths)
    else:
        chain_stopped = True
        current_partial = {'stage': stage, 'resume': str(out)}
        break

_write_chain_state(completed_paths, partial=current_partial)
todo = [s for s in CFG['stages_to_run'] if s not in completed_paths]
if chain_stopped:
    print(f'\nCHAIN PAUSED at the wall budget. Remaining stages: {todo}. '
          f'Re-run this notebook (with its own output attached as an input) to continue.',
          flush=True)
else:
    print('\nCHAIN COMPLETE: all selected stages finished.', flush=True)
print('stage outputs:', {k: str(v) for k, v in STAGE_OUTPUTS.items()})

In [ ]:
### Export the value head as a plain distance model (the beam-search checkpoint) ###
V_ONLY_PATH = None
EVAL_LABEL = None


def _existing(p):
    return p is not None and Path(p).exists()


if 'az' in STAGE_OUTPUTS and _existing(STAGE_OUTPUTS['az']):
    az_ckpt = STAGE_OUTPUTS['az']
    V_ONLY_PATH = export_dual_head(az_ckpt, MODELS_DIR / 'az_v_only.pt', head='value')
    export_dual_head(az_ckpt, MODELS_DIR / 'az_pi_only.pt', head='policy')
    EVAL_LABEL = f'this run ({Path(az_ckpt).name})'
else:
    # Evaluate the most advanced stage output whose file is actually present (stale
    # chain-state pointers from older runs may reference files no output carries).
    for _stage in [s for s in STAGE_ORDER if s in STAGE_OUTPUTS][::-1]:
        if _existing(STAGE_OUTPUTS[_stage]):
            V_ONLY_PATH = STAGE_OUTPUTS[_stage]
            EVAL_LABEL = f'this run ({_stage})'
            break

if V_ONLY_PATH is None and REFERENCE_V_ONLY.exists():
    V_ONLY_PATH = REFERENCE_V_ONLY
    EVAL_LABEL = 'shipped reference m_az_v4_v_only'
    print('no freshly trained checkpoint available to evaluate - '
          'using the shipped reference instead')

print('model under evaluation:', V_ONLY_PATH, f'({EVAL_LABEL})')

In [ ]:
### Value-calibration canary ###
# Fast checks that predict whether a V model will work in beam search. Thresholds come
# from our experiment history - several architectures matched training loss but failed
# ALL of them in beam search.
#
#   1. V(solved) ~ 0 and V(depth-1) ~ 1 (exact anchors).
#   2. Exact-distance MAE on the BFS-d6 shell (d = 0..6).
#   3. SATURATION: on deep random walks V must flatten near the puzzle's effective
#      diameter (~25-32), NOT keep growing with walk depth (V@80 - V@40 <= ~10) and
#      NOT collapse (V@80 >= ~20).
#   4. Mid-depth V std (walk depth 15-25) should stay small (~<4): high variance there
#      predicted beam collapse even when the means looked fine.

CANARY_RESULTS = {}


@torch.no_grad()
def run_canary(v_model, label):
    v_model.eval()
    res = {}
    v0 = float(v_model(SOLVED_T.unsqueeze(0)).item())
    d1_states = apply_all_generators(SOLVED_T.unsqueeze(0), GENERATORS_T).squeeze(0)
    v1 = float(v_model(d1_states).mean().item())
    res['V_solved'] = v0
    res['V_d1'] = v1

    # Exact-distance calibration on a BFS-d6 sample.
    if BFS_D6_PATH.exists():
        b_states, b_dists = load_bfs6()
        n = min(CFG['eval']['canary_bfs_samples'], b_states.size(0))
        gen = torch.Generator().manual_seed(0)
        idx = torch.randperm(b_states.size(0), generator=gen)[:n]
        s = b_states[idx].to(DEVICE)
        d = b_dists[idx]
        preds = []
        for i in range(0, n, 8192):
            preds.append(v_model(s[i:i + 8192]).float().cpu())
        pred = torch.cat(preds)
        mae_by_d = {}
        for depth in range(0, 7):
            m = d == depth
            if int(m.sum()) > 0:
                mae_by_d[depth] = float((pred[m] - d[m]).abs().mean())
        res['bfs_mae_by_depth'] = mae_by_d

    # Saturation probe on deep random walks.
    ws, wd = generate_walks_torch(PUZZLE, n_walks=1500, k_max=80,
                                  seed=CFG['seed'] + 999, device=DEVICE, n_back=1)
    preds = []
    for i in range(0, ws.size(0), 8192):
        preds.append(v_model(ws[i:i + 8192]).float())
    pv = torch.cat(preds)
    wd = wd.float()

    def bucket(lo, hi):
        m = (wd >= lo) & (wd < hi)
        return pv[m]

    v40 = float(bucket(35, 45).mean())
    v80 = float(bucket(75, 81).mean())
    std20 = float(bucket(15, 25).std())
    res.update(V_at_d40=v40, V_at_d80=v80, sat_gap=v80 - v40, V_std_d20=std20)

    print(f'--- canary: {label} ---')
    print(f'  V(solved) = {v0:+.3f}   (want ~0)     '
          f'{"PASS" if abs(v0) < 0.5 else "WARN"}')
    print(f'  V(d=1)    = {v1:+.3f}   (want ~1)     '
          f'{"PASS" if 0.5 < v1 < 1.5 else "WARN"}')
    if 'bfs_mae_by_depth' in res:
        mae_str = '  '.join(f'd{k}:{v:.2f}' for k, v in res['bfs_mae_by_depth'].items())
        print(f'  exact-distance MAE: {mae_str}')
    print(f'  V@walk40 = {v40:.1f}  V@walk80 = {v80:.1f}  gap = {v80 - v40:+.1f}  '
          f'{"PASS" if (v80 - v40) <= 10 else "WARN: V grows with walk depth - beam will likely fail"}')
    print(f'  V@walk80 level: {"PASS" if 20 <= v80 <= 40 else "WARN: outside [20,40] - drifting or collapsed"}')
    print(f'  V std @ walk 15-25 = {std20:.2f}  '
          f'{"PASS" if std20 < 4 else "WARN: high mid-depth variance predicts beam collapse"}')
    CANARY_RESULTS[label] = res
    return res


if CFG['eval']['canary'] and V_ONLY_PATH is not None:
    run_canary(load_v_model(V_ONLY_PATH), EVAL_LABEL)
    if (CFG['eval']['compare_reference'] and REFERENCE_V_ONLY.exists()
            and Path(V_ONLY_PATH) != REFERENCE_V_ONLY):
        run_canary(load_v_model(REFERENCE_V_ONLY), 'reference m_az_v4_v_only')

# Beam-search benchmark (cayleypy)

Sanity-check the exported value head by actually solving a few puzzles with cayleypy beam
search (`beam_mode='iterated'`, non-backtracking history) - the same harness as the other
community baseline notebooks, so numbers are directly comparable *within this harness*.

Two honest caveats:

- Our headline results (51/51, mean 87.5 on the stratified sample; full-1001 production
  runs) come from our own solver stack (multi-pass beam at width 16k->65k, a 4-rotation
  symmetry ensemble, and min-merging) - absolute path lengths from a single cayleypy beam
  at width 4096 will be longer. Use this cell to compare checkpoints against each other
  and against other models in the same harness, not to reproduce the headline numbers.
- Never pair this value head with a Q-shortlister distilled from a DIFFERENT value model
  (the shortlist ordering is calibrated to its teacher's V landscape; mixing costs ~5-10%
  path length in our measurements).

In [ ]:
### cayleypy graph helper + small benchmark ###
_CAYLEY = {}


def get_cayley_graph():
    if 'graph' not in _CAYLEY:
        from cayleypy import CayleyGraphDef, CayleyGraph
        with open(PUZZLE_JSON, encoding='utf-8') as f:
            info = json.load(f)
        gens_names = list(info['generators'].keys())
        graph_def = CayleyGraphDef.create(
            generators=[np.array(info['generators'][n]) for n in gens_names],
            generator_names=gens_names,
            central_state=np.array(info['central_state']))
        _CAYLEY['graph'] = CayleyGraph(graph_def, dtype=torch.int8,
                                       bit_encoding_width=None,
                                       batch_size=2 ** 16, hash_chunk_size=2 ** 16)
    return _CAYLEY['graph']


def beam_solve_pid(pid, model, beam_width, max_steps):
    """Beam-search one test puzzle; returns (path_string or None, length or None)."""
    from cayleypy import Predictor
    graph = get_cayley_graph()
    initial = np.array([int(x) for x in TEST_ROWS[pid]['initial_state'].split(',')])
    res = graph.beam_search(
        start_state=initial, beam_width=beam_width, max_steps=max_steps,
        predictor=Predictor(graph, model), beam_mode='iterated',
        history_depth=CFG['eval']['history_depth'], hashed_neigbourhood=0,
        memory_cleanup=False, return_path=True, verbose=0, path_device='cuda')
    if not res.path_found:
        return None, None
    path_str = res.get_path_as_string()
    # Independent verification with our own puzzle code.
    final = PUZZLE.apply_path(tuple(initial), PUZZLE.parse_path(path_str))
    assert PUZZLE.is_solved(final), f'pid {pid}: cayleypy path does not verify!'
    return path_str, len(path_str.split('.'))


BENCH_RESULTS = {}
if CFG['eval']['bench'] and V_ONLY_PATH is not None:
    try:
        import cayleypy  # noqa: F401
        _have_cayleypy = True
    except Exception as e:
        _have_cayleypy = False
        print('cayleypy not available - skipping bench:', e)
    if _have_cayleypy:
        bench_model = load_v_model(V_ONLY_PATH)
        bench_model.eval()
        bw = CFG['eval']['bench_beam_width']
        print(f'benchmark: beam_width={bw}, max_steps={CFG["eval"]["bench_max_steps"]}, '
              f'model={EVAL_LABEL}')
        for pid in CFG['eval']['bench_pids']:
            t0 = time.time()
            path_str, length = beam_solve_pid(pid, bench_model, bw,
                                              CFG['eval']['bench_max_steps'])
            BENCH_RESULTS[pid] = length
            print(f'  pid {pid:4d} | solved: {length is not None} | '
                  f'len: {length} | {time.time() - t0:.1f}s', flush=True)
        solved = [v for v in BENCH_RESULTS.values() if v is not None]
        if solved:
            print(f'bench: {len(solved)}/{len(BENCH_RESULTS)} solved, '
                  f'mean len {sum(solved) / len(solved):.1f}')

In [ ]:
### Optional: solve puzzles and write submission.csv ###
# Off by default (CFG['solve']['enabled']). Solves `list_states_to_solve` (short list by
# default; set to [] to solve ALL 1001 - hours at useful beam widths). Unsolved or
# un-attempted puzzles fall back to the sample submission's path.
SOLVE_STATS = None

_solve_ok = CFG['solve']['enabled']
if _solve_ok:
    try:
        import cayleypy  # noqa: F401
    except Exception as _e:
        print('cayleypy not available - skipping solve:', _e)
        _solve_ok = False

if _solve_ok:
    import pandas as pd

    ckpt = CFG['solve']['checkpoint']
    if ckpt in ('auto', None):
        ckpt = V_ONLY_PATH if V_ONLY_PATH is not None else REFERENCE_V_ONLY
    assert ckpt is not None and Path(ckpt).exists(), f'no solve checkpoint available: {ckpt}'
    solve_model = load_v_model(ckpt)
    solve_model.eval()
    print(f'solving with {ckpt}')

    df_sample = pd.read_csv(SAMPLE_SUB, index_col='initial_state_id')
    todo = CFG['solve']['list_states_to_solve'] or list(range(len(df_sample)))
    print(f"solving {len(todo)} puzzles at beam_width={CFG['solve']['beam_width']}")

    new_paths = dict(df_sample['path'])  # pid -> path, prefilled with sample fallback
    n_solved = 0
    n_improved = 0
    t_all = time.time()
    for i, pid in enumerate(todo):
        t0 = time.time()
        path_str, length = beam_solve_pid(pid, solve_model,
                                          CFG['solve']['beam_width'],
                                          CFG['solve']['max_steps'])
        sample_len = len(str(df_sample['path'].loc[pid]).split('.'))
        if length is not None:
            n_solved += 1
            if length < sample_len:
                new_paths[pid] = path_str
                n_improved += 1
        if i < 20 or pid % 50 == 0:
            print(f'  pid {pid:4d} | solved {length is not None} | len {length} '
                  f'(sample {sample_len}) | {time.time() - t0:.1f}s', flush=True)

    df_out = df_sample.copy()
    df_out['path'] = [new_paths[pid] for pid in df_out.index]
    df_out.to_csv(WORK / 'submission.csv')
    total = int(df_out['path'].apply(lambda x: len(str(x).split('.'))).sum())
    SOLVE_STATS = {'attempted': len(todo), 'solved': n_solved,
                   'improved_vs_sample': n_improved, 'total_moves': total,
                   'wall_min': (time.time() - t_all) / 60}
    print(f'\nsolved {n_solved}/{len(todo)} attempted ({n_improved} beat the sample path)')
    print(f'submission.csv written - total moves over all 1001 rows: {total:,}')
    print('(rows outside list_states_to_solve keep the sample path - '
          'solve everything with list_states_to_solve=[] before submitting for real)')

In [ ]:
### Run summary ###
summary = {
    'chain_complete': not chain_stopped,
    'stages_remaining': [s for s in CFG['stages_to_run'] if s not in completed_paths],
    'stages_trained': {k: str(v) for k, v in STAGE_OUTPUTS.items()},
    'model_under_eval': str(V_ONLY_PATH),
    'eval_label': EVAL_LABEL,
    'canary': CANARY_RESULTS,
    'bench_path_lengths': BENCH_RESULTS,
    'solve': SOLVE_STATS,
    'cfg': CFG,
}
with open(WORK / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, default=str)

print('=' * 70)
print('RUN SUMMARY')
print('=' * 70)
if summary['chain_complete']:
    print('  chain: COMPLETE')
else:
    print(f"  chain: PAUSED at wall budget - re-run to continue; "
          f"remaining: {summary['stages_remaining']}")
for k, v in summary['stages_trained'].items():
    print(f'  trained {k}: {v}')
print(f'  evaluated: {EVAL_LABEL}')
for label, res in CANARY_RESULTS.items():
    print(f"  canary [{label}]: V(solved)={res['V_solved']:+.3f} "
          f"V(d1)={res['V_d1']:+.3f} sat_gap={res['sat_gap']:+.1f} "
          f"V@80={res['V_at_d80']:.1f} std@20={res['V_std_d20']:.2f}")
if BENCH_RESULTS:
    print(f'  bench: {BENCH_RESULTS}')
if SOLVE_STATS:
    print(f'  solve: {SOLVE_STATS}')
print('full details in run_summary.json / cfg.json')

# Notes, pitfalls, and what to try

Things we learned building AZ v4 (each of these cost us GPU-days - read before changing
the config):

1. **Select checkpoints by beam quality, never by training loss.** The az stage's value
   head peaks around epoch 24 and then degrades while both losses keep falling. The same
   holds for very long V-model training in general: a 184-epoch stage-4 variant had lower
   loss than the 50-epoch one and solved 0/20 of the puzzles the 50-epoch one solved.
2. **Batch sizes are part of the recipe.** Halving `rw_batch_size`/`policy_batch_size`
   quadruples gradient steps per epoch: the policy memorizes (top-1 90%+) and the value
   head is destroyed. If you change them, re-tune the stopping epoch from the printed
   top-1 accuracy (sweet spot was ~28%).
3. **Bigger trunks are not free wins.** A 20.5M trunk under this exact recipe was worse
   than the 6M one at every checkpoint. Also, warm-starting across a shape change silently
   skips mismatched layers (you would be fine-tuning a mostly random net) - retrain from
   stage 1 at the new shape instead.
4. **The saturation canary is load-bearing.** Every architecture we tried whose V kept
   growing on deep walks (instead of flattening at ~25-32) failed beam search regardless
   of how good its training loss and shallow calibration looked. Check the canary output
   before spending hours on beam runs.
5. **Better policy data converges faster.** AZ v4 was originally trained on a 76,304-move
   merged solutions set; this notebook ships a stronger public 73,731 set. More-consistent
   paths mean fewer ambiguous (state, action) pairs, so expect the sweet spot at a
   somewhat different epoch - watch top-1 accuracy and compare a couple of checkpoints
   with the bench cell.
6. **Ideas that did NOT work** (so you can skip them): symmetry/rotation augmentation on
   the value head; training the value head on solver paths only (beam expands off-path
   children the model has never seen - random-walk coverage is essential); pairing this V
   with a Q-shortlister distilled from a different V.
7. **Non-ResMLP backbones are UNVALIDATED under this recipe.** The model pool lets you
   push `PilgrimAttnRes` (or your own model) through all five stages, but every
   hyperparameter default here was tuned on the ResMLP. In our broader architecture
   survey, seven different encoder families matched the ResMLP on training loss and then
   failed the saturation canary and/or beam search - so run the canary cell before
   spending hours, and expect to retune per stage.

Improving on AZ v4's value head is genuinely hard - most of our subsequent gains came
from inference (wider beams, symmetry ensembles, min-merging across runs), not from
better models. If you find a config that beats it on the bench cell, please share it in
the competition discussion!

*Assets dataset: [megaminx-az4-training-assets](https://www.kaggle.com/datasets/artgor/megaminx-az4-training-assets).
Structure inspired by @ogurtsov's
[modular baselines notebook](https://www.kaggle.com/code/ogurtsov/cayleypy-rw-modelbaselines-megaminx-adamw).
Happy training!*